In [ ]:
!pip install opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV Version:", cv2.__version__)
print("NumPy Version:", np.__version__)

print("Environment ready!")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Environment ready!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

image_folder = "/content/sample_data/DRONE PHOTOS"

os.makedirs(image_folder, exist_ok=True)

for filename, content in uploaded.items():
    file_path = os.path.join(image_folder, filename)

    with open(file_path, "wb") as f:
        f.write(content)

print("Images saved successfully!")
print(os.listdir(image_folder))

In [ ]:
import glob

image_paths = glob.glob(image_folder + "/*")

print("Number of images found:", len(image_paths))

In [ ]:
images = {}

for path in image_paths:

    filename = os.path.basename(path)

    img = cv2.imread(path)

    if img is not None:
        images[filename] = img
        print(filename, "->", img.shape)

    else:
        print("Failed to load:", filename)

In [ ]:
for filename, img in images.items():

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(9, 5))
    plt.imshow(img_rgb)
    plt.title(filename)
    plt.axis("off")
    plt.show()

In [ ]:
# Find the top-view image
top_filename = [name for name in images.keys()
                if "top" in name.lower()][0]

top_img = images[top_filename]

print("Selected image:", top_filename)
print("Image dimensions:", top_img.shape)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(top_img, cv2.COLOR_BGR2RGB))
plt.title("Pluto Drone - Top View")
plt.axis("off")
plt.show()

In [ ]:
height, width, channels = top_img.shape

print("Height:", height)
print("Width:", width)
print("Color channels:", channels)

In [ ]:
gray_img = cv2.cvtColor(top_img, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(10, 7))
plt.imshow(gray_img, cmap="gray")
plt.title("Grayscale Pluto Drone")
plt.axis("off")
plt.show()

In [ ]:
blur_img = cv2.GaussianBlur(gray_img, (5, 5), 0)

plt.figure(figsize=(10, 7))
plt.imshow(blur_img, cmap="gray")
plt.title("Gaussian Blurred Image")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(top_img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gray_img, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(blur_img, cmap="gray")
plt.title("Gaussian Blur")
plt.axis("off")

plt.show()

In [ ]:
edges = cv2.Canny(blur_img, 50, 150)

plt.figure(figsize=(10, 7))
plt.imshow(edges, cmap="gray")
plt.title("Canny Edge Detection")
plt.axis("off")
plt.show()

In [ ]:
thresholds = [
    (30, 100),
    (50, 150),
    (100, 200)
]

plt.figure(figsize=(15, 5))

for i, (low, high) in enumerate(thresholds):

    edges = cv2.Canny(blur_img, low, high)

    plt.subplot(1, 3, i + 1)
    plt.imshow(edges, cmap="gray")
    plt.title(f"Threshold: {low}, {high}")
    plt.axis("off")

plt.show()

In [ ]:
# Apply Canny
edges = cv2.Canny(blur_img, 50, 150)

# Find contours
contours, hierarchy = cv2.findContours(
    edges,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

print("Total contours detected:", len(contours))

In [ ]:
contour_img = top_img.copy()

cv2.drawContours(
    contour_img,
    contours,
    -1,
    (0, 255, 0),
    2
)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
plt.title("Detected Contours")
plt.axis("off")
plt.show()

In [ ]:
filtered_img = top_img.copy()

for contour in contours:

    area = cv2.contourArea(contour)

    if area > 100:
        cv2.drawContours(
            filtered_img,
            [contour],
            -1,
            (0, 255, 0),
            2
        )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(filtered_img, cv2.COLOR_BGR2RGB))
plt.title("Contours After Area Filtering")
plt.axis("off")
plt.show()

In [ ]:
height, width = top_img.shape[:2]

# Approximate propeller regions
regions = {
    "Propeller 1": (0.18, 0.03, 0.42, 0.42),
    "Propeller 2": (0.55, 0.03, 0.82, 0.42),
    "Propeller 3": (0.18, 0.55, 0.42, 0.98),
    "Propeller 4": (0.55, 0.55, 0.82, 0.98)
}

roi_img = top_img.copy()

for name, (x1, y1, x2, y2) in regions.items():

    x1, x2 = int(x1 * width), int(x2 * width)
    y1, y2 = int(y1 * height), int(y2 * height)

    cv2.rectangle(
        roi_img,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    cv2.putText(
        roi_img,
        name,
        (x1, y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB))
plt.title("Four Propeller ROI Regions")
plt.axis("off")
plt.show()

In [ ]:
propeller_crops = {}

for name, (x1, y1, x2, y2) in regions.items():

    x1, x2 = int(x1 * width), int(x2 * width)
    y1, y2 = int(y1 * height), int(y2 * height)

    crop = top_img[y1:y2, x1:x2]

    propeller_crops[name] = crop

    plt.figure(figsize=(5, 4))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis("off")
    plt.show()

In [ ]:
height, width = top_img.shape[:2]

print("Image width:", width)
print("Image height:", height)

plt.figure(figsize=(14, 9))

plt.imshow(cv2.cvtColor(top_img, cv2.COLOR_BGR2RGB))

plt.xticks(np.arange(0, width, 50))
plt.yticks(np.arange(0, height, 50))

plt.grid(True, alpha=0.4)
plt.title("Pluto Drone - Pixel Coordinate Reference")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

plt.show()

In [ ]:
regions = {
    "Propeller 1": (215, 30, 375, 220),
    "Propeller 2": (510, 30, 710, 220),
    "Propeller 3": (265, 350, 400, 575),
    "Propeller 4": (560, 340, 710, 560)
}

roi_img = top_img.copy()

for name, (x1, y1, x2, y2) in regions.items():

    cv2.rectangle(
        roi_img,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        3
    )

    cv2.putText(
        roi_img,
        name,
        (x1, y1 - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB))
plt.title("Improved Propeller ROI Detection")
plt.axis("off")
plt.show()

In [ ]:
propeller_crops = {}

for name, (x1, y1, x2, y2) in regions.items():

    crop = top_img[y1:y2, x1:x2]

    propeller_crops[name] = crop

    plt.figure(figsize=(5, 4))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis("off")
    plt.show()

In [ ]:
def detect_propellers(image):

    height, width = image.shape[:2]

    regions = {
        "Propeller_1": (215, 30, 375, 220),
        "Propeller_2": (510, 30, 710, 220),
        "Propeller_3": (265, 350, 400, 575),
        "Propeller_4": (560, 340, 710, 560)
    }

    propeller_crops = {}
    detected_image = image.copy()

    for name, (x1, y1, x2, y2) in regions.items():

        crop = image[y1:y2, x1:x2]

        propeller_crops[name] = crop

        cv2.rectangle(
            detected_image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            detected_image,
            name,
            (x1, y1 - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    return propeller_crops, detected_image

In [ ]:
propeller_crops, detected_image = detect_propellers(top_img)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(detected_image, cv2.COLOR_BGR2RGB))
plt.title("Propeller Detection Output")
plt.axis("off")
plt.show()

print("Detected regions:", len(propeller_crops))
print("Region names:", list(propeller_crops.keys()))

In [ ]:
import os
import cv2

output_folder = "/content/propeller_dataset"

os.makedirs(output_folder, exist_ok=True)

for name, crop in propeller_crops.items():

    save_path = os.path.join(output_folder, name + ".jpg")

    cv2.imwrite(save_path, crop)

    print("Saved:", save_path)

In [ ]:
def detect_propellers(image):

    regions = {
        "Propeller_1": (215, 30, 375, 220),
        "Propeller_2": (510, 30, 710, 220),
        "Propeller_3": (265, 350, 400, 575),
        "Propeller_4": (560, 340, 710, 560)
    }

    crops = {}
    detected_image = image.copy()

    for name, (x1, y1, x2, y2) in regions.items():

        crops[name] = image[y1:y2, x1:x2]

        cv2.rectangle(
            detected_image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

    return {
        "propeller_crops": crops,
        "coordinates": regions,
        "annotated_image": detected_image
    }

In [ ]:
result = detect_propellers(top_img)

print("Detected:", len(result["propeller_crops"]))
print("Coordinates:", result["coordinates"])

In [ ]:
from google.colab import files

new_upload = files.upload()

In [ ]:
filename = list(new_upload.keys())[0]

test_img = cv2.imdecode(
    np.frombuffer(new_upload[filename], np.uint8),
    cv2.IMREAD_COLOR
)

print("New image dimensions:", test_img.shape)

plt.figure(figsize=(6, 7))
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
plt.title("New Pluto Drone Image")
plt.axis("off")
plt.show()

In [ ]:
filename = list(new_upload.keys())[0]

test_img = cv2.imdecode(
    np.frombuffer(new_upload[filename], np.uint8),
    cv2.IMREAD_COLOR
)

print("Image dimensions:", test_img.shape)

plt.figure(figsize=(6, 7))
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
plt.title("Second Pluto Drone Image")
plt.axis("off")
plt.show()

In [ ]:
test_result = detect_propellers(test_img)

plt.figure(figsize=(7, 8))
plt.imshow(
    cv2.cvtColor(test_result["annotated_image"], cv2.COLOR_BGR2RGB)
)
plt.title("Detection on Second Image")
plt.axis("off")
plt.show()

In [ ]:
hsv = cv2.cvtColor(test_img, cv2.COLOR_BGR2HSV)

plt.figure(figsize=(7, 7))
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")
plt.show()

print("HSV conversion completed!")

In [ ]:
lower_dark = np.array([0, 0, 0])
upper_dark = np.array([180, 255, 90])

mask = cv2.inRange(hsv, lower_dark, upper_dark)

plt.figure(figsize=(7, 8))
plt.imshow(mask, cmap="gray")
plt.title("Dark Object Segmentation")
plt.axis("off")
plt.show()

In [ ]:
kernel = np.ones((5, 5), np.uint8)

clean_mask = cv2.morphologyEx(
    mask,
    cv2.MORPH_OPEN,
    kernel
)

plt.figure(figsize=(7, 8))
plt.imshow(clean_mask, cmap="gray")
plt.title("Cleaned Dark Object Mask")
plt.axis("off")
plt.show()

In [ ]:
contours, hierarchy = cv2.findContours(
    clean_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

print("Total objects detected:", len(contours))

In [ ]:
contour_img = test_img.copy()

for contour in contours:

    area = cv2.contourArea(contour)

    if area > 100:
        x, y, w, h = cv2.boundingRect(contour)

        cv2.rectangle(
            contour_img,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            contour_img,
            f"Area: {int(area)}",
            (x, max(y - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (0, 255, 0),
            1
        )

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
plt.title("Connected Object Detection")
plt.axis("off")
plt.show()

In [ ]:
contour_img = test_img.copy()

for contour in contours:

    area = cv2.contourArea(contour)

    if area > 100:
        x, y, w, h = cv2.boundingRect(contour)

        cv2.rectangle(
            contour_img,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            contour_img,
            f"Area: {int(area)}",
            (x, max(y - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (0, 255, 0),
            1
        )

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
plt.title("Connected Object Detection")
plt.axis("off")
plt.show()

In [ ]:
kernel_7 = np.ones((7, 7), np.uint8)
kernel_11 = np.ones((11, 11), np.uint8)
kernel_15 = np.ones((15, 15), np.uint8)

opened_7 = cv2.morphologyEx(
    clean_mask,
    cv2.MORPH_OPEN,
    kernel_7
)

opened_11 = cv2.morphologyEx(
    clean_mask,
    cv2.MORPH_OPEN,
    kernel_11
)

opened_15 = cv2.morphologyEx(
    clean_mask,
    cv2.MORPH_OPEN,
    kernel_15
)

In [ ]:
plt.figure(figsize=(15, 8))

plt.subplot(1, 3, 1)
plt.imshow(opened_7, cmap="gray")
plt.title("7 x 7 Kernel")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(opened_11, cmap="gray")
plt.title("11 x 11 Kernel")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(opened_15, cmap="gray")
plt.title("15 x 15 Kernel")
plt.axis("off")

plt.show()

In [ ]:
for name, mask_img in [
    ("7x7", opened_7),
    ("11x11", opened_11),
    ("15x15", opened_15)
]:

    count, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(mask_img)
    )

    print(name, "-> Objects detected:", count - 1)

In [ ]:
# Convert image to grayscale
gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)

# Apply Gaussian blur
blur = cv2.GaussianBlur(gray, (7, 7), 0)

# Apply thresholding
_, body_mask = cv2.threshold(
    blur,
    70,
    255,
    cv2.THRESH_BINARY_INV
)

# Display the result
plt.figure(figsize=(7, 8))
plt.imshow(body_mask, cmap="gray")
plt.title("Drone Body - Initial Threshold")
plt.axis("off")
plt.show()

In [ ]:
body_contours, _ = cv2.findContours(
    body_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# Find largest contour
largest_contour = max(
    body_contours,
    key=cv2.contourArea
)

# Draw largest contour
body_result = test_img.copy()

cv2.drawContours(
    body_result,
    [largest_contour],
    -1,
    (0, 255, 0),
    3
)

plt.figure(figsize=(7, 8))
plt.imshow(cv2.cvtColor(body_result, cv2.COLOR_BGR2RGB))
plt.title("Largest Contour")
plt.axis("off")
plt.show()

In [ ]:
# Find all contours
contours, _ = cv2.findContours(
    body_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# Display contours that may represent the body
shape_img = test_img.copy()

for contour in contours:

    area = cv2.contourArea(contour)

    if area > 5000:

        x, y, w, h = cv2.boundingRect(contour)

        aspect_ratio = w / h

        print(
            "Area:", int(area),
            "Width:", w,
            "Height:", h,
            "Aspect ratio:", round(aspect_ratio, 2)
        )

        cv2.rectangle(
            shape_img,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

plt.figure(figsize=(7, 8))
plt.imshow(cv2.cvtColor(shape_img, cv2.COLOR_BGR2RGB))
plt.title("Large Object Bounding Boxes")
plt.axis("off")
plt.show()

In [ ]:
motor_points = {
    "Motor_1": (115, 235),
    "Motor_2": (550, 150),
    "Motor_3": (155, 660),
    "Motor_4": (620, 610)
}

motor_img = test_img.copy()

for name, (x, y) in motor_points.items():

    cv2.circle(
        motor_img,
        (x, y),
        8,
        (0, 255, 0),
        -1
    )

    cv2.putText(
        motor_img,
        name,
        (x + 10, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        2
    )

plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(motor_img, cv2.COLOR_BGR2RGB))
plt.title("Approximate Motor Center Locations")
plt.axis("off")
plt.show()

In [ ]:
motor_roi_img = test_img.copy()

for name, (x, y) in motor_points.items():

    cv2.rectangle(
        motor_roi_img,
        (x - 65, y - 65),
        (x + 65, y + 65),
        (0, 255, 0),
        2
    )

plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(motor_roi_img, cv2.COLOR_BGR2RGB))
plt.title("Motor-Based Propeller Regions")
plt.axis("off")
plt.show()

In [ ]:
print("Image shape:", test_img.shape)

height, width = test_img.shape[:2]

print("Width:", width)
print("Height:", height)

In [ ]:
plt.figure(figsize=(9, 11))

plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))

plt.xticks(np.arange(0, width, 25))
plt.yticks(np.arange(0, height, 25))

plt.grid(True, alpha=0.4)
plt.title("Motor Center Coordinate Mapping")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

plt.show()

In [ ]:
motor_points = {
    "Motor_1": (60, 100),
    "Motor_2": (285, 55),
    "Motor_3": (75, 320),
    "Motor_4": (325, 290)
}

motor_img = test_img.copy()

for name, (x, y) in motor_points.items():

    cv2.circle(
        motor_img,
        (x, y),
        6,
        (0, 255, 0),
        -1
    )

    cv2.putText(
        motor_img,
        name,
        (x + 8, y - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.4,
        (0, 255, 0),
        1
    )

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(motor_img, cv2.COLOR_BGR2RGB))
plt.title("Four Motor Center Locations")
plt.axis("off")
plt.show()

In [ ]:
motor_roi_img = test_img.copy()

for name, (x, y) in motor_points.items():

    cv2.rectangle(
        motor_roi_img,
        (max(0, x - 55), max(0, y - 55)),
        (min(width, x + 55), min(height, y + 55)),
        (0, 255, 0),
        2
    )

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(motor_roi_img, cv2.COLOR_BGR2RGB))
plt.title("Motor-Based Propeller ROIs")
plt.axis("off")
plt.show()

In [ ]:
def extract_propeller_rois(image, motor_points, box_size=110):

    height, width = image.shape[:2]

    crops = {}
    annotated = image.copy()
    coordinates = {}

    half = box_size // 2

    for name, (cx, cy) in motor_points.items():

        x1 = max(0, cx - half)
        y1 = max(0, cy - half)

        x2 = min(width, cx + half)
        y2 = min(height, cy + half)

        crops[name] = image[y1:y2, x1:x2]

        coordinates[name] = (x1, y1, x2, y2)

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.circle(
            annotated,
            (cx, cy),
            4,
            (0, 0, 255),
            -1
        )

    return crops, coordinates, annotated

In [ ]:
crops, coordinates, annotated = extract_propeller_rois(
    test_img,
    motor_points,
    box_size=110
)

print("Number of propeller regions:", len(crops))
print("Coordinates:", coordinates)

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("Extracted Propeller Regions")
plt.axis("off")
plt.show()

In [ ]:
for name, crop in crops.items():

    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis("off")
    plt.show()

In [ ]:
crops, coordinates, annotated = extract_propeller_rois(
    test_img,
    motor_points,
    box_size=140
)

for name, crop in crops.items():

    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis("off")
    plt.show()

In [ ]:
result = {
    "propeller_crops": crops,
    "coordinates": coordinates,
    "annotated_image": annotated,
    "number_detected": len(crops)
}

In [ ]:
import os
import cv2

output_folder = "/content/propeller_dataset"

os.makedirs(output_folder, exist_ok=True)

for name, crop in crops.items():

    path = os.path.join(output_folder, name + ".jpg")

    cv2.imwrite(path, crop)

    print("Saved:", name)

In [ ]:
print(os.listdir(output_folder))

In [ ]:
def propeller_extraction(image, motor_points, box_size=140):

    height, width = image.shape[:2]

    crops = {}
    coordinates = {}
    annotated = image.copy()

    half = box_size // 2

    for name, (cx, cy) in motor_points.items():

        x1 = max(0, cx - half)
        y1 = max(0, cy - half)

        x2 = min(width, cx + half)
        y2 = min(height, cy + half)

        crops[name] = image[y1:y2, x1:x2]

        coordinates[name] = (x1, y1, x2, y2)

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

    return {
        "propeller_crops": crops,
        "coordinates": coordinates,
        "annotated_image": annotated,
        "number_detected": len(crops)
    }

In [ ]:
result = propeller_extraction(
    test_img,
    motor_points,
    box_size=140
)

print("Number of regions:", result["number_detected"])
print("Available crops:", list(result["propeller_crops"].keys()))

In [ ]:
gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)

gray_blur = cv2.GaussianBlur(
    gray, (5, 5), 1
)

circles = cv2.HoughCircles(
    gray_blur,
    cv2.HOUGH_GRADIENT,
    dp=1.2,
    minDist=40,
    param1=100,
    param2=30,
    minRadius=5,
    maxRadius=25
)

circle_img = test_img.copy()

if circles is not None:

    circles = np.uint16(np.around(circles))

    for x, y, r in circles[0, :]:

        cv2.circle(
            circle_img,
            (x, y),
            r,
            (0, 255, 0),
            2
        )

        cv2.circle(
            circle_img,
            (x, y),
            2,
            (0, 0, 255),
            3
        )

    print("Circles detected:", len(circles[0]))

else:
    print("No circles detected")

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(circle_img, cv2.COLOR_BGR2RGB))
plt.title("Hough Circle Detection")
plt.axis("off")
plt.show()

In [ ]:
# Convert image to grayscale
gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)

# Create ORB detector
orb = cv2.ORB_create(nfeatures=500)

# Detect keypoints and descriptors
keypoints, descriptors = orb.detectAndCompute(gray, None)

# Draw detected keypoints
orb_img = cv2.drawKeypoints(
    test_img,
    keypoints,
    None,
    color=(0, 255, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

print("Total keypoints detected:", len(keypoints))

plt.figure(figsize=(8, 9))
plt.imshow(cv2.cvtColor(orb_img, cv2.COLOR_BGR2RGB))
plt.title("ORB Feature Detection")
plt.axis("off")
plt.show()

In [ ]:
# Convert both images to grayscale
gray1 = cv2.cvtColor(top_img, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)

# Create ORB detector
orb = cv2.ORB_create(nfeatures=1000)

# Detect features and descriptors
kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)

print("Reference keypoints:", len(kp1))
print("Test image keypoints:", len(kp2))

# Match descriptors
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

matches = bf.match(des1, des2)

# Sort matches by distance
matches = sorted(matches, key=lambda x: x.distance)

# Display best 40 matches
match_img = cv2.drawMatches(
    top_img, kp1,
    test_img, kp2,
    matches[:40],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
plt.title("ORB Feature Matching")
plt.axis("off")
plt.show()

print("Total matches:", len(matches))

In [ ]:
# Create ORB matcher
bf = cv2.BFMatcher(cv2.NORM_HAMMING)

# Find two nearest matches
knn_matches = bf.knnMatch(des1, des2, k=2)

# Apply Lowe's ratio test
good_matches = []

for pair in knn_matches:

    if len(pair) == 2:
        m, n = pair

        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

print("Original matches:", len(matches))
print("Good matches:", len(good_matches))

In [ ]:
good_match_img = cv2.drawMatches(
    top_img, kp1,
    test_img, kp2,
    good_matches[:50],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(good_match_img, cv2.COLOR_BGR2RGB))
plt.title("ORB - Good Feature Matches")
plt.axis("off")
plt.show()

In [ ]:
if len(good_matches) >= 4:

    src_pts = np.float32([
        kp1[m.queryIdx].pt for m in good_matches
    ]).reshape(-1, 1, 2)

    dst_pts = np.float32([
        kp2[m.trainIdx].pt for m in good_matches
    ]).reshape(-1, 1, 2)

    H, inlier_mask = cv2.findHomography(
        src_pts,
        dst_pts,
        cv2.RANSAC,
        5.0
    )

    if H is not None:

        inliers = [
            m for m, keep in zip(good_matches, inlier_mask.ravel())
            if keep
        ]

        print("Reliable matches:", len(inliers))
        print("Rejected matches:", len(good_matches) - len(inliers))

        inlier_img = cv2.drawMatches(
            top_img, kp1,
            test_img, kp2,
            inliers[:50],
            None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        plt.figure(figsize=(16, 8))
        plt.imshow(cv2.cvtColor(inlier_img, cv2.COLOR_BGR2RGB))
        plt.title("ORB Matches After RANSAC")
        plt.axis("off")
        plt.show()

    else:
        print("Could not estimate transformation.")

else:
    print("Not enough good matches for RANSAC.")

In [ ]:
import os

base_path = "/content/drone_dataset"

folders = [
    "images/correct",
    "images/missing_propeller",
    "images/incorrect_type",
    "images/unclear"
]

for folder in folders:
    os.makedirs(
        os.path.join(base_path, folder),
        exist_ok=True
    )

print("Dataset folders created successfully!")

for root, dirs, files in os.walk(base_path):
    print(root, "->", len(files), "images")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

save_folder = "/content/drone_dataset/images/correct"

os.makedirs(save_folder, exist_ok=True)

for filename, content in uploaded.items():

    save_path = os.path.join(save_folder, filename)

    with open(save_path, "wb") as f:
        f.write(content)

    print("Saved:", filename)

In [ ]:
import os

folder = "/content/drone_dataset/images/correct"

images = [
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Correct images collected:", len(images))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import math

folder = "/content/drone_dataset/images/correct"

image_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

cols = 4
rows = math.ceil(len(image_files) / cols)

plt.figure(figsize=(16, rows * 4))

for i, filename in enumerate(image_files):

    img = cv2.imread(os.path.join(folder, filename))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(filename, fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os

base = "/content/drone_dataset/images"

folders = [
    "correct/top_view",
    "correct/front_side",
    "correct/underside",
    "correct/closeup"
]

for folder in folders:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("Viewpoint folders created!")

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Images uploaded:", len(uploaded))

In [ ]:
import os

base_path = "/content/drone_dataset/images"

for root, dirs, files in os.walk(base_path):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if image_files:
        print(f"\nFolder: {root}")
        print("Images:", len(image_files))
        for name in image_files:
            print(" -", name)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import math

folder = "/content/drone_dataset/images/incorrect"

image_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

cols = 3
rows = math.ceil(len(image_files) / cols)

plt.figure(figsize=(15, rows * 4))

for i, filename in enumerate(image_files):

    img = cv2.imread(os.path.join(folder, filename))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(filename)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os

incorrect_folder = "/content/drone_dataset/images/incorrect"

os.makedirs(incorrect_folder, exist_ok=True)

for filename, content in uploaded.items():

    save_path = os.path.join(incorrect_folder, filename)

    with open(save_path, "wb") as f:
        f.write(content)

    print("Saved:", filename)

print("\nTotal incorrect images:", len(uploaded))

In [ ]:
print(os.listdir(incorrect_folder))

In [ ]:
import cv2
import matplotlib.pyplot as plt

files = [
    f for f in os.listdir(incorrect_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for filename in files:
    img = cv2.imread(os.path.join(incorrect_folder, filename))

    plt.figure(figsize=(7, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(filename)
    plt.axis("off")
    plt.show()

In [ ]:
import os

correct_folder = "/content/drone_dataset/images/correct"
incorrect_folder = "/content/drone_dataset/images/incorrect"

def count_images(folder):
    if not os.path.exists(folder):
        return 0

    return len([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

print("Correct images:", count_images(correct_folder))
print("Incorrect images:", count_images(incorrect_folder))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import math

folder = "/content/drone_dataset/images/incorrect"

files = sorted([
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

plt.figure(figsize=(15, 12))

for i, filename in enumerate(files):
    img = cv2.imread(os.path.join(folder, filename))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(filename)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install ultralytics

In [ ]:
import torch
import ultralytics

print("YOLO version:", ultralytics.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable GPU in Runtime → Change runtime type → T4 GPU")

In [ ]:
import os

base_dir = "/content/drone_dataset/images"

for folder in ["correct", "incorrect"]:
    path = os.path.join(base_dir, folder)

    files = [
        f for f in os.listdir(path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f"{folder}: {len(files)} images")

In [ ]:
import os

base = "/content/drone_dataset/images"

for folder in ["correct", "incorrect"]:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("Dataset folders recreated!")

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

folder = "/content/drone_dataset/images/correct"

for name, data in uploaded.items():
    with open(os.path.join(folder, name), "wb") as f:
        f.write(data)

print("Correct images saved:", len(uploaded))

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

folder = "/content/drone_dataset/images/incorrect"

for name, data in uploaded.items():
    with open(os.path.join(folder, name), "wb") as f:
        f.write(data)

print("Correct images saved:", len(uploaded))

In [ ]:
import os

base = "/content/drone_dataset/images"

for folder in ["correct", "incorrect"]:
    path = os.path.join(base, folder)
    images = [
        f for f in os.listdir(path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    print(f"{folder}: {len(images)} images")

In [ ]:
!pip install labelImg

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import os

os.rename(
    "/content/weights (1).pt",
    "/content/weights.pt"
)

print("Model ready:", os.path.exists("/content/weights.pt"))

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/weights.pt")

print("YOLO model loaded successfully!")

In [ ]:
!pip install ultralytics


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/weights.pt")

print("YOLO model loaded successfully!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import cv2
import matplotlib.pyplot as plt

image_path = next(iter(uploaded.keys()))

results = model.predict(
    source=image_path,
    conf=0.25
)

annotated_image = results[0].plot()

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

print("Number of propellers detected:", len(results[0].boxes))

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

os.rename("/content/weights (1).pt", "/content/weights.pt")

print("Model restored!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

model = YOLO("/content/weights.pt")

image_path = next(iter(uploaded.keys()))

results = model.predict(
    source=image_path,
    conf=0.25
)

annotated = results[0].plot()

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

print("Detected propellers:", len(results[0].boxes))

In [ ]:
for i, box in enumerate(results[0].boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    confidence = float(box.conf[0])

    print(f"Propeller {i+1}:")
    print(f"Coordinates: {x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}")
    print(f"Confidence: {confidence:.2%}")

In [ ]:
motor_positions = {}

for box in results[0].boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    if cy < 200 and cx < 200:
        position = "Top_Left"
    elif cy < 200 and cx >= 200:
        position = "Top_Right"
    elif cy >= 200 and cx < 200:
        position = "Bottom_Left"
    else:
        position = "Bottom_Right"

    motor_positions[position] = (cx, cy)

print(motor_positions)

In [ ]:
import cv2
import matplotlib.pyplot as plt

image = cv2.imread(image_path)
output = image.copy()

for position, (cx, cy) in motor_positions.items():

    cx, cy = int(cx), int(cy)

    cv2.circle(output, (cx, cy), 6, (0, 0, 255), -1)

    cv2.putText(
        output,
        position,
        (cx - 35, cy - 12),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (0, 255, 0),
        1
    )

plt.figure(figsize=(9, 9))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
motor_reference = {
    "Motor_1": {
        "position": "FL",
        "rotation": "CW"
    },
    "Motor_2": {
        "position": "FR",
        "rotation": "CCW"
    },
    "Motor_3": {
        "position": "RR",
        "rotation": "CW"
    },
    "Motor_4": {
        "position": "RL",
        "rotation": "CCW"
    }
}

for motor, details in motor_reference.items():
    print(motor, details)

In [ ]:
# Map image quadrants to motor numbers
position_to_motor = {
    "Top_Left": "Motor_1",
    "Top_Right": "Motor_2",
    "Bottom_Right": "Motor_3",
    "Bottom_Left": "Motor_4"
}

detected_motors = {}

for position, coordinates in motor_positions.items():

    motor = position_to_motor[position]

    detected_motors[motor] = {
        "position": motor_reference[motor]["position"],
        "rotation": motor_reference[motor]["rotation"],
        "coordinates": coordinates
    }

for motor, details in detected_motors.items():
    print(motor, details)

In [ ]:
def verify_drone_installation(results):

    detected_count = len(results[0].boxes)

    if detected_count != 4:
        return {
            "status": "UNABLE TO VERIFY",
            "detected": detected_count,
            "expected": 4,
            "message": "Missing or undetected propeller."
        }

    motor_status = {}

    for position, motor in position_to_motor.items():

        motor_status[motor] = {
            "position": motor_reference[motor]["position"],
            "expected_rotation": motor_reference[motor]["rotation"],
            "detected": True
        }

    return {
        "status": "DETECTION COMPLETE",
        "motor_status": motor_status,
        "message": "All four motor positions detected. Rotation not visually verified."
    }


report = verify_drone_installation(results)

print("INSTALLATION REPORT")
print("-------------------")
print("Status:", report["status"])
print("Message:", report["message"])

for motor, details in report.get("motor_status", {}).items():
    print(motor, details)

In [ ]:
def verify_drone_installation(results, confidence_threshold=0.50):

    boxes = results[0].boxes

    # Filter detections by confidence
    valid_boxes = [
        box for box in boxes
        if float(box.conf[0]) >= confidence_threshold
    ]

    detected_count = len(valid_boxes)

    if detected_count < 4:
        return {
            "status": "UNABLE TO VERIFY",
            "detected": detected_count,
            "message": "One or more propellers could not be reliably detected."
        }

    if detected_count > 4:
        return {
            "status": "UNABLE TO VERIFY",
            "detected": detected_count,
            "message": "Extra detections found. Manual inspection required."
        }

    return {
        "status": "DETECTION COMPLETE",
        "detected": detected_count,
        "message": "All four propellers detected. CW/CCW installation remains unverified."
    }


report = verify_drone_installation(results)

print("INSTALLATION REPORT")
print("-------------------")
print(report)

In [ ]:
def verify_motor_positions(results, confidence_threshold=0.50):

    # Expected motor positions
    expected_positions = [
        "Top_Left",
        "Top_Right",
        "Bottom_Left",
        "Bottom_Right"
    ]

    # Store detected motor positions
    detected_positions = {}

    boxes = results[0].boxes

    # Check each YOLO detection
    for box in boxes:

        confidence = float(box.conf[0])

        # Ignore low-confidence detections
        if confidence < confidence_threshold:
            continue

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        # Calculate center coordinates
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        # Identify image quadrant
        if cy < results[0].orig_shape[0] / 2:
            vertical = "Top"
        else:
            vertical = "Bottom"

        if cx < results[0].orig_shape[1] / 2:
            horizontal = "Left"
        else:
            horizontal = "Right"

        position = f"{vertical}_{horizontal}"

        # Store detection
        detected_positions.setdefault(position, []).append({
            "center": (round(cx, 2), round(cy, 2)),
            "confidence": round(confidence, 3)
        })

    # Check all four quadrants
    missing = [
        p for p in expected_positions
        if p not in detected_positions
    ]

    duplicate = [
        p for p, detections in detected_positions.items()
        if len(detections) > 1
    ]

    # Generate result
    if missing or duplicate:

        status = "UNABLE TO VERIFY"

    else:

        status = "DETECTION COMPLETE"

    return {
        "status": status,
        "detected_positions": detected_positions,
        "missing_positions": missing,
        "duplicate_positions": duplicate
    }


report = verify_motor_positions(results)

print("MOTOR VERIFICATION REPORT")
print("=========================")
print("Status:", report["status"])
print("Detected positions:", list(report["detected_positions"].keys()))
print("Missing positions:", report["missing_positions"])
print("Duplicate positions:", report["duplicate_positions"])

In [ ]:
def generate_installation_report(results):

    report = verify_motor_positions(results)

    print("=" * 40)
    print("PLUTO DRONE INSTALLATION REPORT")
    print("=" * 40)

    print("\nDetection Status:", report["status"])

    print("\nMotor Position Details:")

    for position in [
        "Top_Left",
        "Top_Right",
        "Bottom_Left",
        "Bottom_Right"
    ]:
        detections = report["detected_positions"].get(position, [])

        if len(detections) == 1:
            print(f"{position}: DETECTED")
        else:
            print(f"{position}: NOT VERIFIED")

    print("\nFinal Installation Status:")
    print("UNABLE TO VERIFY")

    print("\nReason:")
    print("CW/CCW propeller installation cannot")
    print("currently be confirmed visually.")

    print("=" * 40)

    return report


final_report = generate_installation_report(results)

In [ ]:
from google.colab import files

uploaded = files.upload()

test_image_path = next(iter(uploaded.keys()))

print("Testing image:", test_image_path)

In [ ]:
from google.colab import files

uploaded = files.upload()

test_image_path = next(iter(uploaded.keys()))

print("Testing image:", test_image_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt

test_results = model.predict(
    source=test_image_path,
    conf=0.25
)

annotated = test_results[0].plot()

plt.figure(figsize=(9, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

print("Number of detected propellers:",
      len(test_results[0].boxes))

In [ ]:
from google.colab import files

uploaded = files.upload()

test_image_path = next(iter(uploaded.keys()))

print("Testing image:", test_image_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt

test_results = model.predict(
    source=test_image_path,
    conf=0.25
)

annotated = test_results[0].plot()

plt.figure(figsize=(9, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

print("Number of detected propellers:",
      len(test_results[0].boxes))

In [ ]:
from google.colab import files

uploaded = files.upload()

test_image_path = next(iter(uploaded.keys()))

print("Testing image:", test_image_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt

test_results = model.predict(
    source=test_image_path,
    conf=0.25
)

annotated = test_results[0].plot()

plt.figure(figsize=(9, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

print("Number of detected propellers:",
      len(test_results[0].boxes))

In [ ]:
import os

os.makedirs("reference_engine", exist_ok=True)

print("Reference engine folder created successfully!")

In [ ]:
%%writefile reference_engine/reference_config.py

REFERENCE_CONFIG = {
    "drone": {
        "model": "PLUTO",
        "configuration_version": "1.0"
    },

    "coordinate_system": {
        "image_origin": "TOP_LEFT",
        "x_axis": "RIGHT",
        "y_axis": "DOWN",
        "front_direction": "UNKNOWN"
    },

    "motors": {
        "FL": {
            "position": "FRONT_LEFT",
            "propeller_type": "UNKNOWN",
            "orientation": "UNKNOWN"
        },
        "FR": {
            "position": "FRONT_RIGHT",
            "propeller_type": "UNKNOWN",
            "orientation": "UNKNOWN"
        },
        "RL": {
            "position": "REAR_LEFT",
            "propeller_type": "UNKNOWN",
            "orientation": "UNKNOWN"
        },
        "RR": {
            "position": "REAR_RIGHT",
            "propeller_type": "UNKNOWN",
            "orientation": "UNKNOWN"
        }
    },

    "verification_status": "DRAFT"
}

In [ ]:
%%writefile reference_engine/reference_geometry.py

import numpy as np

REFERENCE_GEOMETRY = {
    "FL": {"x": -1.0, "y": 1.0},
    "FR": {"x": 1.0, "y": 1.0},
    "RL": {"x": -1.0, "y": -1.0},
    "RR": {"x": 1.0, "y": -1.0}
}


def calculate_drone_center(geometry):

    points = np.array([
        [data["x"], data["y"]]
        for data in geometry.values()
    ], dtype=float)

    return np.mean(points, axis=0)


def validate_geometry(geometry):

    required = {"FL", "FR", "RL", "RR"}
    errors = []

    if not isinstance(geometry, dict):
        return {"valid": False, "errors": ["Invalid geometry."]}

    if set(geometry.keys()) != required:
        return {
            "valid": False,
            "errors": ["Incorrect motor positions."]
        }

    points = {}

    for motor, data in geometry.items():

        try:
            point = np.array(
                [data["x"], data["y"]],
                dtype=float
            )
        except (KeyError, TypeError, ValueError):
            errors.append(f"{motor}: Invalid coordinates.")
            continue

        if not np.all(np.isfinite(point)):
            errors.append(f"{motor}: Non-finite coordinates.")
        else:
            points[motor] = point

    if errors:
        return {"valid": False, "errors": errors}

    if not (
        points["FL"][1] > points["RL"][1]
        and points["FR"][1] > points["RR"][1]
        and points["FL"][0] < points["FR"][0]
        and points["RL"][0] < points["RR"][0]
    ):
        errors.append("Motor geometry has invalid orientation.")

    center = np.mean(list(points.values()), axis=0)

    if not np.allclose(center, [0, 0], atol=1e-6):
        errors.append("Geometry is not centred at origin.")

    return {
        "valid": len(errors) == 0,
        "errors": errors
    }

In [ ]:
%%writefile reference_engine/reference_validation.py

REQUIRED_MOTORS = {"FL", "FR", "RL", "RR"}

EXPECTED_POSITIONS = {
    "FL": "FRONT_LEFT",
    "FR": "FRONT_RIGHT",
    "RL": "REAR_LEFT",
    "RR": "REAR_RIGHT"
}


def validate_reference_config(config):

    errors = []

    if not isinstance(config, dict):
        return {
            "valid": False,
            "errors": ["Invalid configuration."]
        }

    motors = config.get("motors")

    if not isinstance(motors, dict):
        return {
            "valid": False,
            "errors": ["Motor data missing."]
        }

    if set(motors.keys()) != REQUIRED_MOTORS:
        errors.append(
            "Exactly FL, FR, RL and RR are required."
        )

    for motor_id, expected_position in EXPECTED_POSITIONS.items():

        if motor_id not in motors:
            continue

        details = motors[motor_id]

        if not isinstance(details, dict):
            errors.append(f"{motor_id}: Invalid motor data.")
            continue

        if details.get("position") != expected_position:
            errors.append(f"{motor_id}: Incorrect position label.")

        if details.get("propeller_type") in [None, "", "UNKNOWN"]:
            errors.append(f"{motor_id}: Propeller type unconfirmed.")

        if details.get("orientation") not in ["CW", "CCW"]:
            errors.append(f"{motor_id}: Orientation unconfirmed.")

    coordinate_system = config.get("coordinate_system")

    if not isinstance(coordinate_system, dict):
        errors.append("Coordinate system missing.")

    elif coordinate_system.get("front_direction") in [
        None, "", "UNKNOWN"
    ]:
        errors.append("Front direction unconfirmed.")

    if config.get("verification_status") != "CONFIRMED":
        errors.append("Reference configuration is not confirmed.")

    return {
        "valid": len(errors) == 0,
        "errors": errors
    }

In [ ]:
%%writefile reference_engine/coordinate_transform.py

import numpy as np


def map_image_coordinates(
    detections,
    front_direction,
    ambiguity_tolerance=1e-6
):

    if len(detections) != 4:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": ["Exactly four motor detections required."]
        }

    ids = [item.get("id") for item in detections]

    if None in ids or len(set(ids)) != 4:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": ["Missing or duplicate motor IDs."]
        }

    try:
        points = np.array([
            item["center"] for item in detections
        ], dtype=float)

        front = np.asarray(front_direction, dtype=float)

    except (KeyError, TypeError, ValueError):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": ["Invalid coordinate input."]
        }

    if points.shape != (4, 2) or not np.all(np.isfinite(points)):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": ["Invalid motor coordinates."]
        }

    magnitude = np.linalg.norm(front)

    if front.shape != (2,) or not np.isfinite(magnitude) or magnitude == 0:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": ["Invalid front direction."]
        }

    center = np.mean(points, axis=0)

    front = front / magnitude
    right = np.array([-front[1], front[0]])

    mapping = {}
    errors = []

    for item, point in zip(detections, points):

        relative = point - center

        x_projection = np.dot(relative, right)
        y_projection = np.dot(relative, front)

        if (
            abs(x_projection) <= ambiguity_tolerance
            or abs(y_projection) <= ambiguity_tolerance
        ):
            errors.append(
                f"{item['id']}: Ambiguous motor position."
            )
            continue

        side = "LEFT" if x_projection < 0 else "RIGHT"
        row = "FRONT" if y_projection > 0 else "REAR"

        mapping[item["id"]] = f"{row}_{side}"

    if errors:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": errors
        }

    if set(mapping.values()) != {
        "FRONT_LEFT",
        "FRONT_RIGHT",
        "REAR_LEFT",
        "REAR_RIGHT"
    }:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "errors": ["Motor geometry is inconsistent."]
        }

    return {
        "status": "MAPPED",
        "drone_center": center.tolist(),
        "mapping": mapping,
        "errors": []
    }

In [ ]:
%%writefile reference_engine/reference_geometry.py

import numpy as np

REFERENCE_GEOMETRY = {
    "FL": {"x": -1.0, "y": 1.0},
    "FR": {"x": 1.0, "y": 1.0},
    "RL": {"x": -1.0, "y": -1.0},
    "RR": {"x": 1.0, "y": -1.0}
}


def calculate_drone_center(geometry):

    points = np.array([
        [data["x"], data["y"]]
        for data in geometry.values()
    ], dtype=float)

    return np.mean(points, axis=0)


def validate_geometry(geometry):

    required = {"FL", "FR", "RL", "RR"}

    if not isinstance(geometry, dict):
        return {"valid": False, "errors": ["Invalid geometry."]}

    if set(geometry.keys()) != required:
        return {"valid": False, "errors": ["Invalid motor positions."]}

    try:
        points = {
            motor: np.array([data["x"], data["y"]], dtype=float)
            for motor, data in geometry.items()
        }
    except (KeyError, TypeError, ValueError):
        return {"valid": False, "errors": ["Invalid coordinates."]}

    errors = []

    if not all(np.all(np.isfinite(p)) for p in points.values()):
        errors.append("Non-finite coordinates.")

    if errors:
        return {"valid": False, "errors": errors}

    if not (
        points["FL"][1] > points["RL"][1]
        and points["FR"][1] > points["RR"][1]
        and points["FL"][0] < points["FR"][0]
        and points["RL"][0] < points["RR"][0]
    ):
        errors.append("Motor geometry orientation invalid.")

    center = np.mean(list(points.values()), axis=0)

    if not np.allclose(center, [0, 0], atol=1e-6):
        errors.append("Geometry is not centred at origin.")

    return {
        "valid": len(errors) == 0,
        "errors": errors
    }

In [ ]:
%%writefile reference_engine/reference_engine.py

from .reference_validation import validate_reference_config
from .reference_geometry import validate_geometry
from .coordinate_transform import map_image_coordinates


POSITION_TO_MOTOR = {
    "FRONT_LEFT": "FL",
    "FRONT_RIGHT": "FR",
    "REAR_LEFT": "RL",
    "REAR_RIGHT": "RR"
}


def run_reference_engine(
    reference_config,
    reference_geometry,
    detected_motors,
    front_direction
):

    # Step 1: Validate reference configuration
    config_check = validate_reference_config(reference_config)

    if not config_check["valid"]:
        return {
            "status": "REFERENCE_INVALID",
            "errors": config_check["errors"]
        }

    # Step 2: Validate reference geometry
    geometry_check = validate_geometry(reference_geometry)

    if not geometry_check["valid"]:
        return {
            "status": "REFERENCE_INVALID",
            "errors": geometry_check["errors"]
        }

    # Step 3: Map detected propellers to motor positions
    mapping = map_image_coordinates(
        detected_motors,
        front_direction
    )

    if mapping["status"] != "MAPPED":
        return {
            "status": "UNABLE_TO_VERIFY",
            "errors": mapping["errors"]
        }

    # Step 4: Convert mapped positions to motor IDs
    motor_mapping = {
        detection_id: POSITION_TO_MOTOR[position]
        for detection_id, position in mapping["mapping"].items()
    }

    return {
        "status": "MAPPING_SUCCESSFUL",
        "drone_center": mapping["drone_center"],
        "motor_mapping": motor_mapping,
        "reference_status": reference_config["verification_status"]
    }

In [ ]:
%%writefile reference_engine/__init__.py

from .reference_config import REFERENCE_CONFIG
from .reference_geometry import REFERENCE_GEOMETRY
from .reference_validation import validate_reference_config
from .coordinate_transform import map_image_coordinates
from .reference_engine import run_reference_engine

In [ ]:
import reference_engine

from reference_engine import (
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    validate_reference_config,
    map_image_coordinates,
    run_reference_engine
)

print("Reference Engine imported successfully!")

print("\nDrone:", REFERENCE_CONFIG["drone"]["model"])
print("Motor positions:", list(REFERENCE_CONFIG["motors"].keys()))

print("\nReference validation:")
print(validate_reference_config(REFERENCE_CONFIG))

In [ ]:
from reference_engine import map_image_coordinates

# Simulated top-view detections
test_detections = [
    {"id": "D1", "center": [100, 100]},  # Front-left
    {"id": "D2", "center": [300, 100]},  # Front-right
    {"id": "D3", "center": [100, 300]},  # Rear-left
    {"id": "D4", "center": [300, 300]}   # Rear-right
]

# Drone front faces the top of the image
front_direction = [0, -1]

result = map_image_coordinates(
    test_detections,
    front_direction
)

print("Coordinate Mapping Result:")
print(result)

In [ ]:
from google.colab import files

uploaded_weights = files.upload()

In [ ]:
print("Uploaded files:", list(uploaded_weights.keys()))

In [ ]:
from ultralytics import YOLO
import os

weights_path = "/content/weights (1).pt"

assert os.path.exists(weights_path), "YOLO weights file not found!"

model = YOLO(weights_path)

print("YOLO model loaded successfully!")
print("Model classes:", model.names)

In [ ]:
from google.colab import files
from IPython.display import display, Image

# Upload a top-view Pluto drone image
uploaded_image = files.upload()

image_path = list(uploaded_image.keys())[0]

# Run YOLO prediction
results = model.predict(
    source=image_path,
    conf=0.25,
    save=True,
    show_labels=True
)

# Display detection results
results[0].show()

# Print bounding box details
print("\nYOLO Detection Results:")

for i, box in enumerate(results[0].boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    confidence = float(box.conf[0])
    class_id = int(box.cls[0])

    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    print(f"\nDetection {i+1}")
    print("Class:", model.names[class_id])
    print("Confidence:", round(confidence, 3))
    print("Center:", [round(center_x, 2), round(center_y, 2)])

In [ ]:
import numpy as np

detections = []

for i, box in enumerate(results[0].boxes):

    x1, y1, x2, y2 = box.xyxy[0].tolist()

    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    detections.append({
        "id": f"D{i+1}",
        "center": [center_x, center_y],
        "bbox": [x1, y1, x2, y2],
        "confidence": float(box.conf[0])
    })

print("Actual YOLO Coordinates:")
for detection in detections:
    print(detection)

In [ ]:
from reference_engine import map_image_coordinates

# Front of drone points towards the top of the image
front_direction = [0, -1]

# Map actual YOLO detections
mapping_result = map_image_coordinates(
    detections,
    front_direction
)

print("ACTUAL YOLO COORDINATE MAPPING")
print("--------------------------------")

print("Status:", mapping_result["status"])
print("Drone center:", mapping_result["drone_center"])

print("\nMotor Mapping:")

for detection_id, position in mapping_result["mapping"].items():
    print(f"{detection_id} -> {position}")

print("\nErrors:", mapping_result["errors"])

In [ ]:
# Expected motor positions
expected_positions = {
    "FRONT_LEFT",
    "FRONT_RIGHT",
    "REAR_LEFT",
    "REAR_RIGHT"
}

# Extract mapped positions
detected_positions = set(
    mapping_result["mapping"].values()
)

# Check missing positions
missing_positions = expected_positions - detected_positions

# Check duplicate assignments
duplicate_mapping = (
    len(mapping_result["mapping"]) != len(detected_positions)
)

print("PLUTO MOTOR POSITION VALIDATION")
print("--------------------------------")

print("Detected positions:", detected_positions)

if mapping_result["status"] == "MAPPED" and not missing_positions and not duplicate_mapping:
    print("\nPOSITION VALIDATION: PASS")
    print("All four motor positions are detected.")
else:
    print("\nPOSITION VALIDATION: FAIL")
    print("Missing positions:", missing_positions)
    print("Duplicate assignments:", duplicate_mapping)

In [ ]:
EXPECTED_POSITIONS = {
    "FRONT_LEFT",
    "FRONT_RIGHT",
    "REAR_LEFT",
    "REAR_RIGHT"
}

def validate_motor_installation(mapping_result):

    detected = list(mapping_result["mapping"].values())
    detected_positions = set(detected)

    missing = EXPECTED_POSITIONS - detected_positions

    duplicates = [
        position for position in detected_positions
        if detected.count(position) > 1
    ]

    if missing or duplicates or mapping_result["status"] != "MAPPED":
        status = "FAIL"
    else:
        status = "PASS"

    return {
        "installation_status": status,
        "detected_count": len(detected),
        "missing_positions": sorted(missing),
        "duplicate_positions": sorted(duplicates),
        "detected_positions": sorted(detected_positions)
    }


# Test actual YOLO mapping
installation_result = validate_motor_installation(mapping_result)

print("PLUTO INSTALLATION VALIDATION")
print("--------------------------------")
print(installation_result)

In [ ]:
test_cases = {
    "ALL_PROPELLERS_PRESENT": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_RIGHT",
        "D3": "REAR_LEFT",
        "D4": "REAR_RIGHT"
    },

    "FRONT_LEFT_MISSING": {
        "D2": "FRONT_RIGHT",
        "D3": "REAR_LEFT",
        "D4": "REAR_RIGHT"
    },

    "FRONT_RIGHT_MISSING": {
        "D1": "FRONT_LEFT",
        "D3": "REAR_LEFT",
        "D4": "REAR_RIGHT"
    },

    "REAR_LEFT_MISSING": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_RIGHT",
        "D4": "REAR_RIGHT"
    },

    "REAR_RIGHT_MISSING": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_RIGHT",
        "D3": "REAR_LEFT"
    }
}

print("MISSING PROPELLER TEST")
print("======================")

for test_name, test_mapping in test_cases.items():

    simulated_result = {
        "status": "MAPPED",
        "mapping": test_mapping
    }

    result = validate_motor_installation(simulated_result)

    print(f"\nTest: {test_name}")
    print("Status:", result["installation_status"])
    print("Missing:", result["missing_positions"])

In [ ]:
test_cases = {
    "DUPLICATE_POSITION": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_LEFT",
        "D3": "REAR_LEFT",
        "D4": "REAR_RIGHT"
    },

    "INVALID_POSITION": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_RIGHT",
        "D3": "REAR_LEFT",
        "D4": "UNKNOWN"
    }
}

for name, test_mapping in test_cases.items():

    result = validate_motor_installation({
        "status": "MAPPED",
        "mapping": test_mapping
    })

    print("\nTest:", name)
    print("Installation:", result["installation_status"])
    print("Missing:", result["missing_positions"])
    print("Duplicates:", result["duplicate_positions"])

In [ ]:
%%writefile reference_engine/reference_config.py

REFERENCE_CONFIG = {
    "drone": {
        "model": "PLUTO",
        "front_direction": "NORTH",
        "configuration_status": "ASSUMED"
    },

    "motors": {
        "FL": {
            "position": "FRONT_LEFT",
            "propeller_type": "A"
        },
        "FR": {
            "position": "FRONT_RIGHT",
            "propeller_type": "B"
        },
        "RL": {
            "position": "REAR_LEFT",
            "propeller_type": "B"
        },
        "RR": {
            "position": "REAR_RIGHT",
            "propeller_type": "A"
        }
    }
}

print("Pluto reference configuration loaded.")

In [ ]:
import importlib
import reference_engine.reference_config as config_module

# Reload updated configuration
importlib.reload(config_module)

REFERENCE_CONFIG = config_module.REFERENCE_CONFIG

print("Configuration keys:", REFERENCE_CONFIG.keys())
print("\nDrone details:", REFERENCE_CONFIG["drone"])

print("\nMotor configuration:")
for motor, details in REFERENCE_CONFIG["motors"].items():
    print(motor, "->", details)

In [ ]:
%%writefile reference_engine/reference_validation.py

def validate_reference_config(config):

    errors = []

    expected_motors = {"FL", "FR", "RL", "RR"}

    motors = config.get("motors", {})
    drone = config.get("drone", {})

    if set(motors.keys()) != expected_motors:
        errors.append("Invalid motor positions.")

    for motor, details in motors.items():

        if details.get("propeller_type") not in ["A", "B"]:
            errors.append(
                f"{motor}: Invalid or unknown propeller type."
            )

        if not details.get("position"):
            errors.append(
                f"{motor}: Motor position missing."
            )

    # Check diagonal arrangement
    if len(motors) == 4:
        if motors["FL"]["propeller_type"] != motors["RR"]["propeller_type"]:
            errors.append("FL and RR propeller types must match.")

        if motors["FR"]["propeller_type"] != motors["RL"]["propeller_type"]:
            errors.append("FR and RL propeller types must match.")

    if drone.get("front_direction") not in ["NORTH", "SOUTH", "EAST", "WEST"]:
        errors.append("Front direction is unconfirmed.")

    return {
        "valid": len(errors) == 0,
        "configuration_status": drone.get("configuration_status", "UNKNOWN"),
        "errors": errors
    }

In [ ]:
import importlib
import reference_engine.reference_validation as validation_module

importlib.reload(validation_module)

result = validation_module.validate_reference_config(
    REFERENCE_CONFIG
)

print("REFERENCE VALIDATION:")
print(result)

In [ ]:
%%writefile reference_engine/propeller_validation.py

def validate_propeller_types(detected_types, reference_config):

    results = {}
    errors = []

    for motor, details in reference_config["motors"].items():

        expected = details["propeller_type"]
        detected = detected_types.get(motor)

        if detected is None:
            status = "UNABLE TO VERIFY"

        elif detected == expected:
            status = "PASS"

        else:
            status = "FAIL"
            errors.append(
                f"{motor}: Expected {expected}, detected {detected}"
            )

        results[motor] = {
            "expected": expected,
            "detected": detected,
            "status": status
        }

    return {
        "results": results,
        "errors": errors
    }

In [ ]:
from reference_engine.propeller_validation import validate_propeller_types

# Simulated correct installation
correct_types = {
    "FL": "A",
    "FR": "B",
    "RL": "B",
    "RR": "A"
}

result = validate_propeller_types(
    correct_types,
    REFERENCE_CONFIG
)

print("CORRECT INSTALLATION TEST")
print(result)

# Simulated incorrect installation
incorrect_types = {
    "FL": "B",
    "FR": "B",
    "RL": "B",
    "RR": "A"
}

result2 = validate_propeller_types(
    incorrect_types,
    REFERENCE_CONFIG
)

print("\nINCORRECT INSTALLATION TEST")
print(result2)

In [ ]:
%%writefile reference_engine/installation_report.py

from .reference_validation import validate_reference_config
from .propeller_validation import validate_propeller_types
from .reference_config import REFERENCE_CONFIG


def generate_installation_report(
    mapping_result,
    detected_types=None
):

    reference_check = validate_reference_config(
        REFERENCE_CONFIG
    )

    position_mapping = mapping_result.get("mapping", {})

    detected_positions = set(position_mapping.values())

    expected_positions = {
        "FRONT_LEFT",
        "FRONT_RIGHT",
        "REAR_LEFT",
        "REAR_RIGHT"
    }

    missing_positions = expected_positions - detected_positions

    duplicate_positions = (
        len(position_mapping) != len(detected_positions)
    )

    position_valid = (
        mapping_result.get("status") == "MAPPED"
        and not missing_positions
        and not duplicate_positions
    )

    propeller_check = None

    if detected_types is not None and position_valid:
        propeller_check = validate_propeller_types(
            detected_types,
            REFERENCE_CONFIG
        )

    if not reference_check["valid"]:
        final_status = "UNABLE TO VERIFY"

    elif not position_valid:
        final_status = "FAIL"

    elif propeller_check is None:
        final_status = "UNABLE TO VERIFY"

    elif propeller_check["errors"]:
        final_status = "FAIL"

    elif any(
        item["status"] == "UNABLE TO VERIFY"
        for item in propeller_check["results"].values()
    ):
        final_status = "UNABLE TO VERIFY"

    else:
        final_status = "PASS"

    return {
        "drone": REFERENCE_CONFIG["drone"]["model"],
        "configuration": REFERENCE_CONFIG["drone"]["configuration_status"],
        "final_status": final_status,
        "position_validation": {
            "valid": position_valid,
            "missing": sorted(missing_positions),
            "duplicates": duplicate_positions
        },
        "propeller_validation": propeller_check
    }

In [ ]:
from reference_engine.installation_report import generate_installation_report

# Test with actual YOLO motor mapping
report = generate_installation_report(
    mapping_result,
    detected_types={
        "FL": "A",
        "FR": "B",
        "RL": "B",
        "RR": "A"
    }
)

print("PLUTO DRONE INSTALLATION REPORT")
print("================================")

for key, value in report.items():
    print(f"\n{key}: {value}")

In [ ]:
from reference_engine.installation_report import generate_installation_report

# Correct motor mapping
correct_mapping = {
    "status": "MAPPED",
    "mapping": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_RIGHT",
        "D3": "REAR_LEFT",
        "D4": "REAR_RIGHT"
    }
}

# Test 1: Missing propeller
missing_mapping = {
    "status": "MAPPED",
    "mapping": {
        "D1": "FRONT_LEFT",
        "D2": "FRONT_RIGHT",
        "D3": "REAR_LEFT"
    }
}

# Test 2: Incorrect propeller type
wrong_types = {
    "FL": "B",
    "FR": "B",
    "RL": "B",
    "RR": "A"
}

# Test 3: Unknown propeller type
unknown_types = {
    "FL": "A",
    "FR": "B",
    "RL": "B",
    "RR": None
}

tests = {
    "CORRECT_INSTALLATION": (correct_mapping, {
        "FL": "A", "FR": "B", "RL": "B", "RR": "A"
    }),

    "MISSING_PROPELLER": (missing_mapping, {
        "FL": "A", "FR": "B", "RL": "B", "RR": "A"
    }),

    "WRONG_PROPELLER_TYPE": (correct_mapping, wrong_types),

    "UNKNOWN_PROPELLER_TYPE": (correct_mapping, unknown_types)
}

for name, (mapping, types) in tests.items():

    result = generate_installation_report(mapping, types)

    print("\nTEST:", name)
    print("FINAL STATUS:", result["final_status"])
    print("POSITION:", result["position_validation"])

    if result["propeller_validation"]:
        print("PROPELLER ERRORS:",
              result["propeller_validation"]["errors"])

In [ ]:
import os
import json

BASE = "/content/reference_dataset"

folders = [
    "complete_drone/top",
    "complete_drone/front",
    "complete_drone/rear",
    "complete_drone/side",
    "individual_propellers/propeller_1",
    "individual_propellers/propeller_2",
    "individual_propellers/propeller_3",
    "individual_propellers/propeller_4"
]

for folder in folders:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

metadata = {
    "drone": "PLUTO",
    "reference_status": "DRAFT",
    "complete_drone_views": [
        "top", "front", "rear", "side"
    ],
    "individual_propellers": [
        "propeller_1",
        "propeller_2",
        "propeller_3",
        "propeller_4"
    ]
}

with open(os.path.join(BASE, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Reference database created successfully!")

In [ ]:
from google.colab import files
import os
import shutil
import json

# Separate external drone testing folder
BASE = "/content/external_drone_test"

os.makedirs(BASE, exist_ok=True)

# Upload external drone image
uploaded = files.upload()

registered = []

for filename in uploaded:

    destination = os.path.join(BASE, filename)

    shutil.copy2(filename, destination)

    registered.append(filename)

# Save external test metadata
metadata = {
    "dataset_type": "EXTERNAL_DRONE_TEST",
    "drone_identity": "NON_PLUTO",
    "purpose": "YOLO generalization testing",
    "expected_propellers": 4,
    "registered_images": registered
}

with open(os.path.join(BASE, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("External drone images registered:", len(registered))

for filename in registered:
    print("SUCCESS:", filename)

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("/content/weights (1).pt")

results = model.predict(
    source="/content/external_drone_test/",
    conf=0.25,
    save=True
)

for result in results:

    print("\nImage:", os.path.basename(result.path))
    print("Detected propellers:", len(result.boxes))

    for i, box in enumerate(result.boxes):
        print(
            f"Propeller {i+1}: "
            f"Confidence = {float(box.conf[0]):.3f}"
        )

In [ ]:
confidences = []

for result in results:
    for box in result.boxes:
        confidences.append(float(box.conf[0]))

print("External Drone Evaluation")
print("-------------------------")
print("Expected propellers: 4")
print("Detected propellers:", len(confidences))
print("Average confidence:",
      round(sum(confidences) / len(confidences), 3))
print("Minimum confidence:", round(min(confidences), 3))
print("Maximum confidence:", round(max(confidences), 3))

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

model = YOLO("/content/weights (1).pt")

TEST_FOLDER = "/content/external_drone_test"

results = model.predict(
    source=TEST_FOLDER,
    conf=0.25,
    save=True
)

evaluation = []

for result in results:

    count = len(result.boxes)

    confidence = (
        result.boxes.conf.cpu().tolist()
        if count > 0 else []
    )

    evaluation.append({
        "Image": os.path.basename(result.path),
        "Expected": 4,
        "Detected": count,
        "Average Confidence": (
            round(sum(confidence) / len(confidence), 3)
            if confidence else 0
        ),
        "Detection Status": (
            "4/4 DETECTED" if count == 4
            else "CHECK DETECTIONS"
        )
    })

df = pd.DataFrame(evaluation)

display(df)

df.to_csv(
    "/content/external_drone_evaluation.csv",
    index=False
)

print("Evaluation report saved successfully.")

In [ ]:
%%writefile /content/reference_engine/drone_identity_validation.py

def validate_drone_identity(
    detected_drone,
    reference_drone="PLUTO"
):
    """
    Validate whether the detected drone identity
    matches the reference drone.
    """

    if detected_drone is None:
        return {
            "status": "UNABLE TO VERIFY",
            "error": "Drone identity is unknown"
        }

    if detected_drone.upper() != reference_drone.upper():
        return {
            "status": "FAIL",
            "error": (
                f"External or incorrect drone detected: "
                f"{detected_drone}"
            )
        }

    return {
        "status": "PASS",
        "error": None
    }

In [ ]:
import sys
sys.path.append("/content")

from reference_engine.drone_identity_validation import (
    validate_drone_identity
)

print("TEST 1: Pluto drone")
print(validate_drone_identity("PLUTO"))

print("\nTEST 2: External drone")
print(validate_drone_identity("DJI"))

print("\nTEST 3: Unknown drone")
print(validate_drone_identity(None))

In [ ]:
%%writefile /content/reference_engine/installation_report.py

from .reference_validation import validate_reference_config
from .drone_identity_validation import validate_drone_identity


def generate_installation_report(
    mapping_result,
    detected_types=None,
    detected_drone=None
):

    identity = validate_drone_identity(detected_drone)

    if identity["status"] != "PASS":
        return {
            "drone_identity": identity,
            "final_status": identity["status"],
            "message": identity["error"]
        }

    config_check = validate_reference_config()

    if not config_check["valid"]:
        return {
            "drone_identity": identity,
            "final_status": "FAIL",
            "message": "Reference configuration is invalid"
        }

    if mapping_result is None:
        return {
            "drone_identity": identity,
            "final_status": "UNABLE TO VERIFY",
            "message": "Motor positions are unavailable"
        }

    # Position validation
    from .reference_engine import run_reference_engine

    position_result = run_reference_engine(mapping_result)

    if position_result.get("status") != "PASS":
        return {
            "drone_identity": identity,
            "position_validation": position_result,
            "final_status": "FAIL",
            "message": "Propeller position validation failed"
        }

    return {
        "drone_identity": identity,
        "position_validation": position_result,
        "final_status": "PASS",
        "message": "Installation verification passed"
    }

In [ ]:
import importlib
import reference_engine.installation_report as report_module

importlib.reload(report_module)

generate_installation_report = report_module.generate_installation_report

import inspect
print(inspect.signature(generate_installation_report))

In [ ]:
%%writefile /content/reference_engine/installation_report.py

from .reference_config import REFERENCE_CONFIG
from .reference_validation import validate_reference_config
from .drone_identity_validation import validate_drone_identity
from .reference_engine import run_reference_engine


def generate_installation_report(
    mapping_result,
    detected_types=None,
    detected_drone=None
):

    # 1. Drone identity validation
    identity = validate_drone_identity(detected_drone)

    if identity["status"] != "PASS":
        return {
            "drone_identity": identity,
            "final_status": identity["status"],
            "message": identity["error"]
        }

    # 2. Reference configuration validation
    config_check = validate_reference_config(REFERENCE_CONFIG)

    if not config_check["valid"]:
        return {
            "drone_identity": identity,
            "final_status": "FAIL",
            "message": "Reference configuration is invalid",
            "errors": config_check.get("errors", [])
        }

    # 3. Mapping availability
    if mapping_result is None:
        return {
            "drone_identity": identity,
            "final_status": "UNABLE TO VERIFY",
            "message": "Motor positions are unavailable"
        }

    # 4. Position validation
    position_result = run_reference_engine(mapping_result)

    if position_result.get("status") != "PASS":
        return {
            "drone_identity": identity,
            "position_validation": position_result,
            "final_status": "FAIL",
            "message": "Propeller position validation failed"
        }

    return {
        "drone_identity": identity,
        "position_validation": position_result,
        "final_status": "PASS",
        "message": "Installation verification passed"
    }

In [ ]:
import importlib
import reference_engine.installation_report as report_module

importlib.reload(report_module)

generate_installation_report = report_module.generate_installation_report

print("Updated function:", generate_installation_report)

In [ ]:
import inspect
import reference_engine.reference_validation as rv
import reference_engine.installation_report as ir

print("REFERENCE VALIDATION FUNCTION")
print("Signature:", inspect.signature(rv.validate_reference_config))
print("Source:")
print(inspect.getsource(rv.validate_reference_config))

print("\nINSTALLATION REPORT FUNCTION")
print("Signature:", inspect.signature(ir.generate_installation_report))
print("Source:")
print(inspect.getsource(ir.generate_installation_report))

In [ ]:
import inspect
import reference_engine.reference_engine as re

print("Function signature:")
print(inspect.signature(re.run_reference_engine))

print("\nFunction source:")
print(inspect.getsource(re.run_reference_engine))

In [ ]:
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_validation import validate_reference_config

print(validate_reference_config(REFERENCE_CONFIG))

In [ ]:
import inspect
from reference_engine.reference_engine import run_reference_engine

print("Signature:")
print(inspect.signature(run_reference_engine))

print("\nSource:")
print(inspect.getsource(run_reference_engine))

In [ ]:
%%writefile /content/reference_engine/installation_report.py

from .reference_config import REFERENCE_CONFIG
from .reference_geometry import REFERENCE_GEOMETRY
from .reference_validation import validate_reference_config
from .drone_identity_validation import validate_drone_identity
from .reference_engine import run_reference_engine


def generate_installation_report(
    detected_motors=None,
    detected_drone=None,
    front_direction=None,
    detected_types=None
):

    # 1. Validate drone identity
    identity = validate_drone_identity(detected_drone)

    if identity["status"] != "PASS":
        return {
            "drone_identity": identity,
            "final_status": identity["status"],
            "message": identity["error"]
        }

    # 2. Validate reference configuration
    config_check = validate_reference_config(REFERENCE_CONFIG)

    if not config_check["valid"]:
        return {
            "final_status": "FAIL",
            "message": "Invalid reference configuration",
            "errors": config_check["errors"]
        }

    # 3. Check input availability
    if detected_motors is None or front_direction is None:
        return {
            "final_status": "UNABLE TO VERIFY",
            "message": "Motor detections or front direction unavailable"
        }

    # 4. Run reference engine
    engine_result = run_reference_engine(
        REFERENCE_CONFIG,
        REFERENCE_GEOMETRY,
        detected_motors,
        front_direction
    )

    if engine_result["status"] != "MAPPING_SUCCESSFUL":
        return {
            "drone_identity": identity,
            "reference_engine": engine_result,
            "final_status": "UNABLE TO VERIFY",
            "message": "Motor mapping could not be completed"
        }

    return {
        "drone_identity": identity,
        "reference_engine": engine_result,
        "final_status": "MAPPING_SUCCESSFUL",
        "message": "Drone identity and motor mapping verified"
    }

In [ ]:
import importlib
import reference_engine.installation_report as report_module

importlib.reload(report_module)

generate_installation_report = report_module.generate_installation_report

print("Installation report module updated.")

In [ ]:
for drone in ["PLUTO", "DJI", None]:

    report = generate_installation_report(
        detected_motors=detections,
        detected_drone=drone,
        front_direction=[0, -1]
    )

    print("\nDrone:", drone)
    print("Final status:", report["final_status"])
    print("Message:", report["message"])

In [ ]:
report = generate_installation_report(
    detected_motors=detections,
    detected_drone="PLUTO",
    front_direction=[0, -1]
)

print("FINAL STATUS:", report["final_status"])
print("\nREFERENCE ENGINE DETAILS:")
print(report.get("reference_engine"))

In [ ]:
from reference_engine.reference_geometry import REFERENCE_GEOMETRY
from reference_engine.reference_validation import validate_reference_config
from reference_engine.reference_geometry import validate_geometry

print("REFERENCE CONFIG:")
print(validate_reference_config(REFERENCE_CONFIG))

print("\nREFERENCE GEOMETRY:")
print(REFERENCE_GEOMETRY)

print("\nGEOMETRY VALIDATION:")
print(validate_geometry(REFERENCE_GEOMETRY))

In [ ]:
from reference_engine.coordinate_transform import map_image_coordinates

mapping_test = map_image_coordinates(
    detections,
    [0, -1]
)

print("MAPPING STATUS:", mapping_test["status"])
print("\nMAPPING DETAILS:")
print(mapping_test)

In [ ]:
from reference_engine.reference_engine import run_reference_engine
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY

engine_test = run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    detections,
    [0, -1]
)

print(engine_test)

In [ ]:
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY

print("REFERENCE CONFIG:")
print(REFERENCE_CONFIG)

print("\nREFERENCE GEOMETRY:")
print(REFERENCE_GEOMETRY)

In [ ]:
import inspect
from reference_engine.reference_validation import validate_reference_config

print(inspect.getsource(validate_reference_config))

In [ ]:
REFERENCE_CONFIG = {
    "drone": {
        "model": "PLUTO",
        "front_direction": "NORTH",
        "configuration_status": "ASSUMED",
        "coordinate_system": "IMAGE"
    },
    "motors": {
        "FL": {
            "position": "FRONT_LEFT",
            "propeller_type": "A",
            "orientation": None
        },
        "FR": {
            "position": "FRONT_RIGHT",
            "propeller_type": "B",
            "orientation": None
        },
        "RL": {
            "position": "REAR_LEFT",
            "propeller_type": "B",
            "orientation": None
        },
        "RR": {
            "position": "REAR_RIGHT",
            "propeller_type": "A",
            "orientation": None
        }
    }
}

In [ ]:
import os

BASE = "/content/reference_engine"

print("REFERENCE ENGINE FILES")
print("==============================")

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for filename in files:
        print(indent + "    " + filename)

In [ ]:
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY

print("REFERENCE CONFIGURATION:")
print(REFERENCE_CONFIG)

print("\nREFERENCE GEOMETRY:")
print(REFERENCE_GEOMETRY)

print("\nREFERENCE DATASET:")
print(os.listdir("/content/reference_dataset"))

In [ ]:
import os
import cv2

BASE = "/content/reference_dataset"

for root, dirs, files in os.walk(BASE):

    for filename in files:

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(root, filename)

            image = cv2.imread(path)

            if image is not None:
                height, width = image.shape[:2]

                print("IMAGE:", filename)
                print("FOLDER:", os.path.relpath(root, BASE))
                print("SIZE:", width, "x", height)
                print("STATUS: LOADED")
                print("----------------------------")

            else:
                print("ERROR:", path)

In [ ]:
print("YOLO DETECTIONS:")
print(detections)

print("\nNUMBER OF DETECTIONS:")
print(len(detections))

In [ ]:
%%writefile /content/reference_engine/propeller_cropping.py

import os
import cv2


def crop_detected_propellers(
    image_path,
    detections,
    output_dir="/content/propeller_crops"
):

    os.makedirs(output_dir, exist_ok=True)

    image = cv2.imread(image_path)

    if image is None:
        raise ValueError(f"Unable to read image: {image_path}")

    height, width = image.shape[:2]

    cropped_results = {}

    for detection in detections:

        detection_id = detection["id"]

        x1, y1, x2, y2 = detection["bbox"]

        # Convert coordinates to integers
        x1 = max(0, int(x1))
        y1 = max(0, int(y1))
        x2 = min(width, int(x2))
        y2 = min(height, int(y2))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        crop_path = os.path.join(
            output_dir,
            f"{detection_id}.jpg"
        )

        cv2.imwrite(crop_path, crop)

        cropped_results[detection_id] = {
            "crop_path": crop_path,
            "bbox": [x1, y1, x2, y2],
            "confidence": detection["confidence"]
        }

    return cropped_results

In [ ]:
from reference_engine.propeller_cropping import crop_detected_propellers

# Enter the original image path used for YOLO detection
image_path = "/content/TOP1.jpeg"

cropped_results = crop_detected_propellers(
    image_path=image_path,
    detections=detections
)

print("CROPPING RESULTS")
print("================")

for motor, details in cropped_results.items():
    print(motor, ":", details["crop_path"])

In [ ]:
from google.colab.patches import cv2_imshow
import cv2

for motor, details in cropped_results.items():

    crop = cv2.imread(details["crop_path"])

    print("Detected propeller:", motor)

    cv2_imshow(crop)

In [ ]:
import os
import cv2
from google.colab.patches import cv2_imshow

REFERENCE_FOLDER = "/content/reference_dataset/individual_propellers"

reference_images = []

for filename in os.listdir(REFERENCE_FOLDER):

    path = os.path.join(REFERENCE_FOLDER, filename)

    if filename.lower().endswith((".jpg", ".jpeg", ".png")):

        image = cv2.imread(path)

        if image is not None:
            reference_images.append((filename, path, image))

print("REFERENCE IMAGES:", len(reference_images))

for filename, path, image in reference_images:

    print("\nReference:", filename)
    print("Dimensions:", image.shape)

    cv2_imshow(image)

In [ ]:
from google.colab import files
import os
import shutil
import json

BASE = "/content/reference_dataset"

uploaded = files.upload()

image_categories = {
    "TOP2.jpeg": "complete_drone/top",
    "FRONT(1).jpeg": "complete_drone/front",
    "BACK1.jpeg": "complete_drone/rear",
    "RS.jpeg": "complete_drone/side",

    "PROPELLER 1.jpeg": "individual_propellers",
    "PROPELLER 2.jpeg": "individual_propellers",
    "PROPELLER3.jpeg": "individual_propellers",
    "PROPELLER 4.jpeg": "individual_propellers"
}

registered = []
unrecognized = []

for filename in uploaded:

    # Match filenames without case sensitivity
    matched_name = next(
        (name for name in image_categories
         if name.lower() == filename.lower()),
        None
    )

    if matched_name:

        folder = image_categories[matched_name]

        destination_folder = os.path.join(BASE, folder)
        os.makedirs(destination_folder, exist_ok=True)

        destination = os.path.join(
            destination_folder, filename
        )

        shutil.copy2(filename, destination)
        registered.append(filename)

    else:
        unrecognized.append(filename)

metadata = {
    "drone": "PLUTO",
    "reference_status": "DRAFT",
    "position_verification": True,
    "appearance_verification": True,
    "registered_images": registered
}

with open(os.path.join(BASE, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("REGISTERED:", len(registered))
print("UNRECOGNIZED:", unrecognized)

In [ ]:
import os
import shutil

SOURCE = "/content/drone without blades2.png"

TEST_FOLDER = "/content/external_drone_test/no_blades"

os.makedirs(TEST_FOLDER, exist_ok=True)

shutil.copy2(
    SOURCE,
    os.path.join(TEST_FOLDER, "drone_without_blades.png")
)

print("No-blade test image registered successfully.")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_path = "/content/pluto_reference_dataset.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/")

REFERENCE_BASE = "/content/pluto_reference_dataset"

print("Reference dataset loaded:", os.path.exists(REFERENCE_BASE))

In [ ]:
import os
import cv2

REFERENCE_BASE = "/content/pluto_reference_dataset"

for root, dirs, files in os.walk(REFERENCE_BASE):
    for filename in files:
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(root, filename)
            image = cv2.imread(path)

            if image is not None:
                print("LOADED:", path, "| SIZE:", image.shape)

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("/content/weights (1).pt")

TOP_REFERENCE = os.path.join(
    REFERENCE_BASE, "complete_drone", "top"
)

reference_results = model.predict(
    source=TOP_REFERENCE,
    conf=0.25,
    save=True
)

for result in reference_results:
    print("\nReference:", os.path.basename(result.path))
    print("Detected propellers:", len(result.boxes))

In [ ]:
from google.colab.patches import cv2_imshow
import cv2
import os

for result in reference_results:

    print("Reference:", os.path.basename(result.path))

    # Generate annotated image
    annotated = result.plot()

    # Display image
    cv2_imshow(annotated)

In [ ]:
reference_detections = {}

for result in reference_results:

    filename = os.path.basename(result.path)

    detections_ref = []

    for i, box in enumerate(result.boxes):

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        detections_ref.append({
            "id": f"D{i+1}",
            "bbox": [x1, y1, x2, y2],
            "center": [
                (x1 + x2) / 2,
                (y1 + y2) / 2
            ],
            "confidence": float(box.conf[0])
        })

    reference_detections[filename] = detections_ref

print("Reference detections stored.")

for name, detections_ref in reference_detections.items():
    print(name, ":", len(detections_ref), "propellers")

In [ ]:
%%writefile /content/reference_engine/verification_reports.py

import json


EXPECTED_MOTORS = ["FL", "FR", "RL", "RR"]


def _get_motor_mapping(engine_result):
    """
    Convert detection-to-motor mapping into motor-to-detection mapping.
    """
    mapping = engine_result.get("motor_mapping", {})

    return {
        motor: detection_id
        for detection_id, motor in mapping.items()
    }


def generate_error_report(
    detections,
    engine_result,
    reference_config,
    appearance_scores=None,
    detected_types=None,
    similarity_threshold=0.70,
    confidence_threshold=0.50
):
    """
    Generate a detailed installation error report.

    appearance_scores:
        {"FL": 0.90, "FR": 0.85, ...}

    detected_types:
        {"FL": "A", "FR": "B", ...}
    """

    appearance_scores = appearance_scores or {}
    detected_types = detected_types or {}

    errors = []
    warnings = []

    motor_mapping = _get_motor_mapping(engine_result)

    detection_lookup = {
        item["id"]: item for item in detections
    }

    # 1. Check detection count
    if len(detections) < 4:
        errors.append({
            "parameter": "Propeller count",
            "expected": 4,
            "detected": len(detections),
            "affected_motor": "UNKNOWN",
            "reason": "One or more propellers were not detected.",
            "severity": "HIGH",
            "recommendation": "Inspect the missing or obstructed propeller."
        })

    elif len(detections) > 4:
        errors.append({
            "parameter": "Propeller count",
            "expected": 4,
            "detected": len(detections),
            "affected_motor": "UNKNOWN",
            "reason": "Additional propeller detections were found.",
            "severity": "HIGH",
            "recommendation": "Inspect possible false-positive detections."
        })

    # 2. Check motor position mapping
    for motor in EXPECTED_MOTORS:

        detection_id = motor_mapping.get(motor)

        if detection_id is None:
            errors.append({
                "parameter": "Motor position",
                "expected": motor,
                "detected": "MISSING",
                "affected_motor": motor,
                "reason": f"No detected propeller was mapped to {motor}.",
                "severity": "HIGH",
                "recommendation": f"Inspect the {motor} motor and propeller."
            })
            continue

        detection = detection_lookup.get(detection_id)

        if detection is None:
            errors.append({
                "parameter": "Motor mapping",
                "expected": motor,
                "detected": detection_id,
                "affected_motor": motor,
                "reason": "Mapped detection ID was not found.",
                "severity": "HIGH",
                "recommendation": "Recheck YOLO detections and coordinate mapping."
            })
            continue

        confidence = float(detection.get("confidence", 0))

        if confidence < confidence_threshold:
            errors.append({
                "parameter": "Detection confidence",
                "expected": f">= {confidence_threshold}",
                "detected": round(confidence, 3),
                "affected_motor": motor,
                "reason": "Detection confidence is below the configured threshold.",
                "severity": "MEDIUM",
                "recommendation": "Capture a clearer image and inspect this propeller."
            })

    # 3. Check appearance similarity
    for motor, score_data in appearance_scores.items():

        score = (
            score_data.get("similarity")
            if isinstance(score_data, dict)
            else score_data
        )

        if score is None:
            continue

        score = float(score)

        if score < similarity_threshold:
            errors.append({
                "parameter": "Propeller appearance",
                "expected": f">= {similarity_threshold:.2f}",
                "detected": round(score, 3),
                "affected_motor": motor,
                "reason": "Propeller appearance differs from its reference.",
                "severity": "HIGH",
                "recommendation": "Inspect blade shape, propeller condition and installation."
            })

    # 4. Check A/B propeller type
    for motor, detected_type in detected_types.items():

        expected_type = (
            reference_config.get("motors", {})
            .get(motor, {})
            .get("propeller_type")
        )

        if detected_type is None:
            continue

        if expected_type != detected_type:
            errors.append({
                "parameter": "Propeller type",
                "expected": expected_type,
                "detected": detected_type,
                "affected_motor": motor,
                "reason": "Detected propeller type differs from the reference configuration.",
                "severity": "HIGH",
                "recommendation": f"Check the propeller type installed at {motor}."
            })

    # 5. Configuration confidence warning
    config_status = reference_config.get(
        "drone", {}
    ).get("configuration_status", "UNKNOWN")

    if config_status != "CONFIRMED":
        warnings.append(
            "Reference configuration is ASSUMED or unconfirmed. "
            "A/B propeller and rotation checks cannot be treated as "
            "manufacturer-verified."
        )

    # 6. Overall result
    if errors:
        final_status = "FAIL"
    elif engine_result.get("status") != "MAPPING_SUCCESSFUL":
        final_status = "UNABLE TO VERIFY"
    elif config_status != "CONFIRMED":
        final_status = "UNABLE TO VERIFY"
    else:
        final_status = "PASS"

    return {
        "report_type": "DETAILED ERROR REPORT",
        "drone": reference_config.get("drone", {}).get("model", "UNKNOWN"),
        "final_status": final_status,
        "total_errors": len(errors),
        "total_warnings": len(warnings),
        "errors": errors,
        "warnings": warnings
    }


def generate_correction_report(
    detections,
    engine_result,
    image_shape,
    reference_centers_normalized,
    reference_config,
    appearance_scores=None,
    similarity_threshold=0.70
):
    """
    Calculate image-coordinate corrections.

    reference_centers_normalized:
        {
            "FL": (x_normalized, y_normalized),
            "FR": (x_normalized, y_normalized),
            "RL": (x_normalized, y_normalized),
            "RR": (x_normalized, y_normalized)
        }

    Normalized coordinates range from 0 to 1.
    """

    height, width = image_shape[:2]

    motor_mapping = _get_motor_mapping(engine_result)

    detection_lookup = {
        item["id"]: item for item in detections
    }

    appearance_scores = appearance_scores or {}

    corrections = []

    for motor in EXPECTED_MOTORS:

        detection_id = motor_mapping.get(motor)

        if detection_id is None:
            corrections.append({
                "motor": motor,
                "status": "NOT DETECTED",
                "correction": "Physical inspection required",
                "reason": "No propeller was mapped to this motor."
            })
            continue

        detection = detection_lookup.get(detection_id)

        if detection is None:
            continue

        if motor not in reference_centers_normalized:
            corrections.append({
                "motor": motor,
                "status": "UNABLE TO CALCULATE",
                "correction": "Reference coordinates unavailable",
                "reason": "No normalized reference position was supplied."
            })
            continue

        # Actual propeller center
        actual_x, actual_y = detection["center"]

        # Convert actual coordinates to normalized coordinates
        actual_x_norm = actual_x / width
        actual_y_norm = actual_y / height

        # Expected reference center
        ref_x_norm, ref_y_norm = reference_centers_normalized[motor]

        # Calculate correction vector
        dx_norm = ref_x_norm - actual_x_norm
        dy_norm = ref_y_norm - actual_y_norm

        dx_pixels = dx_norm * width
        dy_pixels = dy_norm * height

        distance_pixels = (
            dx_pixels ** 2 + dy_pixels ** 2
        ) ** 0.5

        distance_percent = (
            distance_pixels /
            ((width ** 2 + height ** 2) ** 0.5)
        ) * 100

        score_data = appearance_scores.get(motor)

        appearance_score = (
            score_data.get("similarity")
            if isinstance(score_data, dict)
            else score_data
        )

        appearance_issue = (
            appearance_score is not None
            and float(appearance_score) < similarity_threshold
        )

        if distance_pixels > 0:
            status = "POSITION DEVIATION"
        else:
            status = "POSITION ALIGNED"

        if appearance_issue:
            correction = (
                "Inspect propeller type, blade shape and physical installation. "
                "Image-coordinate correction cannot fix appearance mismatch."
            )
        elif distance_pixels > 0:
            correction = (
                "Align the propeller with the reference position, "
                "after confirming both images have matching orientation "
                "and perspective."
            )
        else:
            correction = "No positional correction indicated."

        corrections.append({
            "motor": motor,
            "status": status,
            "actual_center_pixels": [
                round(actual_x, 2),
                round(actual_y, 2)
            ],
            "expected_center_normalized": [
                round(ref_x_norm, 4),
                round(ref_y_norm, 4)
            ],
            "correction_x_pixels": round(dx_pixels, 2),
            "correction_y_pixels": round(dy_pixels, 2),
            "correction_distance_pixels": round(distance_pixels, 2),
            "correction_distance_percent": round(distance_percent, 2),
            "appearance_similarity": appearance_score,
            "recommended_action": correction
        })

    return {
        "report_type": "CORRECTION REPORT",
        "drone": reference_config.get("drone", {}).get("model", "UNKNOWN"),
        "coordinate_unit": "pixels",
        "physical_measurement_available": False,
        "corrections": corrections
    }


def print_verification_reports(error_report, correction_report):

    print("\n" + "=" * 55)
    print("DETAILED INSTALLATION ERROR REPORT")
    print("=" * 55)

    print("Drone:", error_report["drone"])
    print("Final status:", error_report["final_status"])
    print("Total errors:", error_report["total_errors"])

    for index, error in enumerate(error_report["errors"], 1):

        print(f"\nERROR {index}")
        print("Parameter:", error["parameter"])
        print("Affected motor:", error["affected_motor"])
        print("Expected:", error["expected"])
        print("Detected:", error["detected"])
        print("Reason:", error["reason"])
        print("Severity:", error["severity"])
        print("Recommendation:", error["recommendation"])

    for warning in error_report["warnings"]:
        print("\nWARNING:", warning)

    print("\n" + "=" * 55)
    print("INSTALLATION CORRECTION REPORT")
    print("=" * 55)

    for item in correction_report["corrections"]:

        print("\nMotor:", item["motor"])
        print("Status:", item["status"])

        if "correction_distance_pixels" in item:
            print("X correction:", item["correction_x_pixels"], "pixels")
            print("Y correction:", item["correction_y_pixels"], "pixels")
            print("Total correction:", item["correction_distance_pixels"], "pixels")
            print("Correction percentage:", item["correction_distance_percent"], "%")

        print("Recommended action:", item.get("recommended_action", item.get("correction")))""

In [ ]:
from reference_engine.coordinate_transform import map_image_coordinates

import cv2
import os

reference_path = (
    "/content/pluto_reference_dataset/"
    "complete_drone/top/top_view_2.png"
)

reference_image = cv2.imread(reference_path)

reference_height, reference_width = reference_image.shape[:2]

# Use YOLO detections already stored for this reference
ref_detections = reference_detections["top_view_2.png"]

reference_mapping = map_image_coordinates(
    ref_detections,
    [0, -1]
)

print("Reference mapping:")
print(reference_mapping)

In [ ]:
POSITION_TO_MOTOR = {
    "FRONT_LEFT": "FL",
    "FRONT_RIGHT": "FR",
    "REAR_LEFT": "RL",
    "REAR_RIGHT": "RR"
}

reference_centers_normalized = {}

for detection_id, position in reference_mapping["mapping"].items():

    motor = POSITION_TO_MOTOR[position]

    detection = next(
        d for d in ref_detections
        if d["id"] == detection_id
    )

    x, y = detection["center"]

    reference_centers_normalized[motor] = (
        x / reference_width,
        y / reference_height
    )

print("Expected reference centers:")
print(reference_centers_normalized)

In [ ]:
from reference_engine.reference_engine import run_reference_engine
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY

engine_result = run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    detections,
    [0, -1]
)

print(engine_result)

In [ ]:
file_path = "/content/reference_engine/verification_reports.py"

with open(file_path, "r") as f:
    code = f.read()

code = code.replace(
    'print("Recommended action:", item.get("recommended_action", item.get("correction")))""',
    'print("Recommended action:", item.get("recommended_action", item.get("correction")))'
)

with open(file_path, "w") as f:
    f.write(code)

print("Syntax correction applied.")

In [ ]:
import py_compile

py_compile.compile(
    "/content/reference_engine/verification_reports.py",
    doraise=True
)

print("SUCCESS: verification_reports.py has valid syntax.")

In [ ]:
from reference_engine.verification_reports import (
    generate_error_report,
    generate_correction_report,
    print_verification_reports
)

print("All three report functions imported successfully.")

In [ ]:
import cv2
import json

from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY
from reference_engine.reference_engine import run_reference_engine

from reference_engine.verification_reports import (
    generate_error_report,
    generate_correction_report,
    print_verification_reports
)

# 1. Run reference engine
engine_result = run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    detections,
    [0, -1]
)

print("ENGINE STATUS:", engine_result["status"])

# 2. Load actual input image
actual_image = cv2.imread("/content/TOP1.jpeg")

if actual_image is None:
    raise ValueError("TOP1.jpeg not found. Check your input image path.")

# 3. Generate detailed error report
error_report = generate_error_report(
    detections=detections,
    engine_result=engine_result,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={},
    detected_types={}
)

# 4. Generate correction report
correction_report = generate_correction_report(
    detections=detections,
    engine_result=engine_result,
    image_shape=actual_image.shape,
    reference_centers_normalized=reference_centers_normalized,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={}
)

# 5. Display both reports
print_verification_reports(error_report, correction_report)

In [ ]:
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY
from reference_engine.reference_validation import validate_reference_config
from reference_engine.reference_engine import run_reference_engine
from reference_engine.coordinate_transform import map_image_coordinates

print("1. REFERENCE CONFIG")
print(validate_reference_config(REFERENCE_CONFIG))

print("\n2. REFERENCE GEOMETRY")
print(REFERENCE_GEOMETRY)

print("\n3. COORDINATE MAPPING")
mapping_result = map_image_coordinates(detections, [0, -1])
print(mapping_result)

print("\n4. COMPLETE REFERENCE ENGINE")
engine_result = run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    detections,
    [0, -1]
)
print(engine_result)

In [ ]:
import inspect
import reference_engine.reference_engine as engine
import reference_engine.reference_validation as validation

print("========== REFERENCE ENGINE SOURCE ==========")
print(inspect.getsource(engine.run_reference_engine))

print("\n========== VALIDATION SOURCE ==========")
print(inspect.getsource(validation))

In [ ]:
import inspect
import reference_engine.reference_engine as engine
import reference_engine.reference_validation as validation

print("ENGINE MODULE:", engine.__file__)

print("\nVALIDATE GEOMETRY USED BY ENGINE:")
print(inspect.getsource(engine.validate_geometry))

print("\nVALIDATE REFERENCE CONFIG USED BY ENGINE:")
print(inspect.getsource(engine.validate_reference_config))

print("\nGEOMETRY VALIDATOR MODULE:")
print(engine.validate_geometry.__module__)

In [ ]:
import reference_engine.reference_engine as engine
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY

print("ENGINE VALIDATOR DETAILS")
print("Name:", engine.validate_reference_config.__name__)
print("Module:", engine.validate_reference_config.__module__)
print("File:", engine.validate_reference_config.__code__.co_filename)
print("First line:", engine.validate_reference_config.__code__.co_firstlineno)

print("\nCONFIG VALIDATION ACTUALLY USED BY ENGINE")
print(engine.validate_reference_config(REFERENCE_CONFIG))

print("\nGEOMETRY VALIDATION ACTUALLY USED BY ENGINE")
print(engine.validate_geometry(REFERENCE_GEOMETRY))

print("\nENGINE FUNCTION GLOBAL VALIDATOR")
print(engine.run_reference_engine.__globals__["validate_reference_config"])

In [ ]:
import inspect
from reference_engine.reference_validation import validate_reference_config
from reference_engine.reference_config import REFERENCE_CONFIG

print("========== ACTUAL VALIDATION FUNCTION ==========")
print(inspect.getsource(validate_reference_config))

print("\n========== CURRENT REFERENCE CONFIG ==========")
print(REFERENCE_CONFIG)

In [ ]:
import importlib
from pathlib import Path

# 1. Fix the incorrect reference-status lookup
engine_path = Path("/content/reference_engine/reference_engine.py")

source = engine_path.read_text()

old_line = 'reference_status": reference_config["verification_status"]'
new_line = '''reference_status": reference_config.get(
            "verification_status",
            reference_config.get("drone", {}).get(
                "configuration_status", "UNKNOWN"
            )
        )'''

if old_line in source:
    source = source.replace(old_line, new_line)
    engine_path.write_text(source)
    print("Engine status lookup fixed.")
else:
    print("Status lookup already fixed or source differs.")

# 2. Reload validation first, then engine
import reference_engine.reference_validation as validation
import reference_engine.reference_engine as engine

importlib.reload(validation)
importlib.reload(engine)

# 3. Verify configuration
from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY

print("\nCONFIG VALIDATION:")
print(engine.validate_reference_config(REFERENCE_CONFIG))

print("\nGEOMETRY VALIDATION:")
print(engine.validate_geometry(REFERENCE_GEOMETRY))

# 4. Run the reference engine again
engine_result = engine.run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    detections,
    [0, -1]
)

print("\nFINAL ENGINE RESULT:")
print(engine_result)

In [ ]:
import cv2
import json

from reference_engine.verification_reports import (
    generate_error_report,
    generate_correction_report,
    print_verification_reports
)

# Load original image
actual_image = cv2.imread("/content/TOP1.jpeg")

if actual_image is None:
    raise FileNotFoundError("TOP1.jpeg not found.")

# Generate detailed error report
error_report = generate_error_report(
    detections=detections,
    engine_result=engine_result,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={},
    detected_types={}
)

# Generate correction report
correction_report = generate_correction_report(
    detections=detections,
    engine_result=engine_result,
    image_shape=actual_image.shape,
    reference_centers_normalized=reference_centers_normalized,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={}
)

# Display both reports
print_verification_reports(
    error_report,
    correction_report
)

In [ ]:
import copy

from reference_engine.reference_engine import run_reference_engine
from reference_engine.verification_reports import generate_error_report

def run_testcase(test_name, test_detections):

    print("\n" + "=" * 55)
    print("TEST CASE:", test_name)
    print("=" * 55)

    result = run_reference_engine(
        REFERENCE_CONFIG,
        REFERENCE_GEOMETRY,
        test_detections,
        [0, -1]
    )

    print("ENGINE STATUS:", result["status"])

    report = generate_error_report(
        detections=test_detections,
        engine_result=result,
        reference_config=REFERENCE_CONFIG,
        appearance_scores={},
        detected_types={}
    )

    print("\nFINAL STATUS:", report["final_status"])
    print("TOTAL ERRORS:", report["total_errors"])

    for error in report.get("errors", []):
        print(
            error.get("parameter"),
            "|",
            error.get("affected_motor"),
            "|",
            error.get("reason")
        )

    return result, report


# TC01: Normal drone
tc01 = copy.deepcopy(detections)

result_01, report_01 = run_testcase(
    "TC01 - NORMAL INSTALLATION",
    tc01
)


# TC02: Remove rear-right propeller
tc02 = [
    d.copy() for d in detections
    if d["id"] != "D4"
]

result_02, report_02 = run_testcase(
    "TC02 - MISSING PROPELLER",
    tc02
)


# TC03: Low confidence detection
tc03 = copy.deepcopy(detections)

for d in tc03:
    if d["id"] == "D2":
        d["confidence"] = 0.30

result_03, report_03 = run_testcase(
    "TC03 - LOW CONFIDENCE",
    tc03
)


# TC04: Simulate positional deviation
tc04 = copy.deepcopy(detections)

for d in tc04:
    if d["id"] == "D4":
        d["center"][0] -= 60
        d["center"][1] -= 50

result_04, report_04 = run_testcase(
    "TC04 - POSITION DEVIATION",
    tc04
)

In [ ]:
import inspect

from reference_engine.coordinate_transform import map_image_coordinates
from reference_engine.verification_reports import (
    generate_error_report,
    generate_correction_report
)

print("========== COORDINATE MAPPING ==========")
print(inspect.getsource(map_image_coordinates))

print("\n========== ERROR REPORT ==========")
print(inspect.getsource(generate_error_report))

print("\n========== CORRECTION REPORT ==========")
print(inspect.getsource(generate_correction_report))

In [ ]:
import copy
from reference_engine.verification_reports import generate_correction_report

tc04 = copy.deepcopy(detections)

# Shift D4 by 60 pixels horizontally and 50 vertically
for d in tc04:
    if d["id"] == "D4":
        d["center"][0] -= 60
        d["center"][1] -= 50

tc04_engine = run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    tc04,
    [0, -1]
)

tc04_correction = generate_correction_report(
    detections=tc04,
    engine_result=tc04_engine,
    image_shape=actual_image.shape,
    reference_centers_normalized=reference_centers_normalized,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={}
)

for item in tc04_correction["corrections"]:
    print(
        item["motor"],
        "|",
        item["status"],
        "|",
        item["correction_distance_pixels"],
        "pixels"
    )

In [ ]:
import inspect
from reference_engine.coordinate_transform import map_image_coordinates
from reference_engine.verification_reports import generate_error_report

print("========== MAPPING FUNCTION ==========")
print(inspect.getsource(map_image_coordinates))

print("\n========== ERROR REPORT FUNCTION ==========")
print(inspect.getsource(generate_error_report))

In [ ]:
import numpy as np
import reference_engine.reference_engine as engine
import reference_engine.coordinate_transform as transform

# --------------------------------------------------
# 1. UPDATED COORDINATE MAPPING
# --------------------------------------------------

def map_image_coordinates_partial(
    detections,
    front_direction,
    ambiguity_tolerance=1e-6
):
    if len(detections) not in (3, 4):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": [
                f"Expected 3 or 4 detections, received {len(detections)}."
            ]
        }

    ids = [item.get("id") for item in detections]

    if None in ids or len(set(ids)) != len(ids):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": ["Missing or duplicate detection IDs."]
        }

    try:
        points = np.array(
            [item["center"] for item in detections],
            dtype=float
        )
        front = np.asarray(front_direction, dtype=float)
    except (KeyError, TypeError, ValueError):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": ["Invalid coordinate input."]
        }

    if points.shape != (len(detections), 2) or not np.all(np.isfinite(points)):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": ["Invalid motor coordinates."]
        }

    magnitude = np.linalg.norm(front)

    if front.shape != (2,) or not np.isfinite(magnitude) or magnitude == 0:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": ["Invalid front direction."]
        }

    center = np.mean(points, axis=0)
    front = front / magnitude
    right = np.array([-front[1], front[0]])

    mapping = {}
    errors = []

    for item, point in zip(detections, points):
        relative = point - center

        x_projection = np.dot(relative, right)
        y_projection = np.dot(relative, front)

        if (
            abs(x_projection) <= ambiguity_tolerance
            or abs(y_projection) <= ambiguity_tolerance
        ):
            errors.append(f"{item['id']}: Ambiguous motor position.")
            continue

        side = "LEFT" if x_projection < 0 else "RIGHT"
        row = "FRONT" if y_projection > 0 else "REAR"

        mapping[item["id"]] = f"{row}_{side}"

    if errors:
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": errors
        }

    if len(set(mapping.values())) != len(mapping):
        return {
            "status": "UNABLE_TO_VERIFY",
            "mapping": {},
            "missing_positions": [],
            "errors": ["Two detections mapped to the same motor position."]
        }

    expected_positions = {
        "FRONT_LEFT",
        "FRONT_RIGHT",
        "REAR_LEFT",
        "REAR_RIGHT"
    }

    missing = sorted(expected_positions - set(mapping.values()))

    return {
        "status": "MAPPED" if len(detections) == 4 else "MAPPED_PARTIAL",
        "drone_center": center.tolist(),
        "mapping": mapping,
        "missing_positions": missing,
        "errors": []
    }


# --------------------------------------------------
# 2. CONNECT UPDATED MAPPER TO REFERENCE ENGINE
# --------------------------------------------------

transform.map_image_coordinates = map_image_coordinates_partial
engine.map_image_coordinates = map_image_coordinates_partial


# --------------------------------------------------
# 3. UPDATED REFERENCE ENGINE
# --------------------------------------------------

def run_reference_engine_partial(
    reference_config,
    reference_geometry,
    detected_motors,
    front_direction
):

    config_check = engine.validate_reference_config(reference_config)

    if not config_check["valid"]:
        return {
            "status": "REFERENCE_INVALID",
            "errors": config_check["errors"]
        }

    geometry_check = engine.validate_geometry(reference_geometry)

    if not geometry_check["valid"]:
        return {
            "status": "REFERENCE_INVALID",
            "errors": geometry_check["errors"]
        }

    mapping = map_image_coordinates_partial(
        detected_motors,
        front_direction
    )

    if mapping["status"] not in ("MAPPED", "MAPPED_PARTIAL"):
        return {
            "status": "UNABLE_TO_VERIFY",
            "errors": mapping["errors"]
        }

    motor_mapping = {
        detection_id: engine.POSITION_TO_MOTOR[position]
        for detection_id, position in mapping["mapping"].items()
    }

    is_partial = mapping["status"] == "MAPPED_PARTIAL"

    return {
        "status": "MAPPING_PARTIAL" if is_partial else "MAPPING_SUCCESSFUL",
        "drone_center": mapping["drone_center"],
        "motor_mapping": motor_mapping,
        "missing_positions": mapping["missing_positions"],
        "reference_status": reference_config.get(
            "drone", {}
        ).get("configuration_status", "UNKNOWN")
    }


engine.run_reference_engine = run_reference_engine_partial

print("Partial detection mapping fix applied.")

In [ ]:
import copy

tc02 = [
    d.copy() for d in detections
    if d["id"] != "D4"
]

tc02_engine = engine.run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    tc02,
    [0, -1]
)

tc02_report = generate_error_report(
    detections=tc02,
    engine_result=tc02_engine,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={},
    detected_types={}
)

print("ENGINE STATUS:", tc02_engine["status"])
print("MOTOR MAPPING:", tc02_engine["motor_mapping"])
print("MISSING POSITIONS:", tc02_engine["missing_positions"])

print("\nERROR REPORT:")
for error in tc02_report["errors"]:
    print(
        error["parameter"],
        "|",
        error["affected_motor"],
        "|",
        error["reason"]
    )

In [ ]:
import copy

def get_d4_correction(test_detections):
    result = engine.run_reference_engine(
        REFERENCE_CONFIG,
        REFERENCE_GEOMETRY,
        test_detections,
        [0, -1]
    )

    report = generate_correction_report(
        detections=test_detections,
        engine_result=result,
        image_shape=actual_image.shape,
        reference_centers_normalized=reference_centers_normalized,
        reference_config=REFERENCE_CONFIG,
        appearance_scores={}
    )

    return next(
        item for item in report["corrections"]
        if item["motor"] == "RR"
    )


original = copy.deepcopy(detections)
shifted = copy.deepcopy(detections)

for d in shifted:
    if d["id"] == "D4":
        d["center"][0] -= 60
        d["center"][1] -= 50

original_result = get_d4_correction(original)
shifted_result = get_d4_correction(shifted)

print("ORIGINAL RR DISTANCE:",
      original_result["correction_distance_pixels"], "pixels")

print("SHIFTED RR DISTANCE:",
      shifted_result["correction_distance_pixels"], "pixels")

print("CHANGE:",
      round(
          shifted_result["correction_distance_pixels"]
          - original_result["correction_distance_pixels"], 2
      ), "pixels")

In [ ]:

# Install CLIP
!pip -q install transformers

import os
import torch
import numpy as np
from PIL import Image
from google.colab import files
from transformers import CLIPProcessor, CLIPModel

# -----------------------------------------
# 1. LOAD CLIP MODEL
# -----------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

# -----------------------------------------
# 2. REFERENCE DATASET
# -----------------------------------------

REFERENCE_ROOT = "/content/drone_reference_dataset"

extensions = (".jpg", ".jpeg", ".png", ".webp")

if not os.path.isdir(REFERENCE_ROOT):
    raise FileNotFoundError(
        f"Reference folder not found: {REFERENCE_ROOT}"
    )

reference_images = {}

for drone_name in os.listdir(REFERENCE_ROOT):

    drone_folder = os.path.join(
        REFERENCE_ROOT, drone_name
    )

    if not os.path.isdir(drone_folder):
        continue

    for root, dirs, filenames in os.walk(drone_folder):

        for filename in filenames:

            if filename.lower().endswith(extensions):

                path = os.path.join(root, filename)

                reference_images.setdefault(
                    drone_name, []
                ).append(path)

if not reference_images:
    raise ValueError("No reference images found.")

print("REFERENCE DATASET")

for drone, paths in reference_images.items():
    print(drone, ":", len(paths), "images")

# -----------------------------------------
# 3. UPLOAD EXTERNAL DRONE IMAGE
# -----------------------------------------

print("\nUpload external drone image:")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No image uploaded.")

input_path = "/content/" + list(uploaded.keys())[0]

input_image = Image.open(input_path).convert("RGB")

# -----------------------------------------
# 4. IMAGE EMBEDDING FUNCTION
# -----------------------------------------

def get_embedding(image):

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    features = features / features.norm(
        dim=-1, keepdim=True
    )

    return features.squeeze(0)

input_embedding = get_embedding(input_image)

# -----------------------------------------
# 5. COMPARE WITH ALL REFERENCES
# -----------------------------------------

drone_scores = {}

for drone, paths in reference_images.items():

    scores = []

    for path in paths:

        try:
            ref_image = Image.open(path).convert("RGB")

            ref_embedding = get_embedding(ref_image)

            similarity = torch.dot(
                input_embedding, ref_embedding
            ).item()

            scores.append(similarity)

        except Exception as e:
            print("Skipped:", path, "|", e)

    if scores:
        # Average of the three highest similarities,
        # or fewer if fewer references are available.
        top_scores = sorted(scores, reverse=True)[:3]

        drone_scores[drone] = float(np.mean(top_scores))

# -----------------------------------------
# 6. RANK DRONE MODELS
# -----------------------------------------

if not drone_scores:
    raise ValueError("Could not compare any reference images.")

ranked = sorted(
    drone_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

best_drone, best_score = ranked[0]

# Similarity is a ranking signal, not a calibrated
# probability or a validated identity threshold.
print("\n" + "=" * 50)
print("DRONE REFERENCE COMPARISON REPORT")
print("=" * 50)

print("Input image:", os.path.basename(input_path))

print("\nMODEL SIMILARITY RANKING")

for drone, score in ranked:
    print(f"{drone}: {score:.4f} ({score * 100:.2f}%)")

print("\nCLOSEST REFERENCE:", best_drone)
print("SIMILARITY SCORE:", round(best_score, 4))

# Do not claim confirmed identity based only on CLIP.
print("\nIDENTITY STATUS: CANDIDATE MATCH")
print("Manual/model-specific verification is required.")

# -----------------------------------------
# 7. SAVE REPORT
# -----------------------------------------

report_path = "/content/drone_comparison_report.txt"

with open(report_path, "w") as f:

    f.write("DRONE REFERENCE COMPARISON REPORT\n")
    f.write("=" * 45 + "\n")
    f.write(f"Input: {input_path}\n")
    f.write(f"Closest reference: {best_drone}\n")
    f.write(f"Similarity: {best_score:.4f}\n")
    f.write("Identity status: CANDIDATE MATCH\n\n")

    for drone, score in ranked:
        f.write(f"{drone}: {score:.4f}\n")

print("\nReport saved:", report_path)

files.download(report_path)


In [ ]:
import os

for folder in [
    "/content/pluto_reference_dataset",
    "/content/reference_dataset",
    "/content/drone_reference_dataset"
]:
    print("\n", folder, ":", os.path.exists(folder))

    if os.path.exists(folder):
        for root, dirs, files_list in os.walk(folder):
            images = [
                f for f in files_list
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            ]

            if images:
                print(root, "->", images)

In [ ]:
REFERENCE_ROOT = "/content/pluto_reference_dataset"

In [ ]:

# ==========================================================
# DRONE IMAGE VERIFICATION: INPUT -> ERROR + CORRECTION REPORT
# ==========================================================

!pip -q install ultralytics

import os
import json
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab import files

from reference_engine.reference_config import REFERENCE_CONFIG
from reference_engine.reference_geometry import REFERENCE_GEOMETRY
from reference_engine.verification_reports import (
    generate_error_report,
    generate_correction_report,
    print_verification_reports
)
import reference_engine.reference_engine as engine

# ----------------------------------------------------------
# 1. LOAD YOLO MODEL
# ----------------------------------------------------------

YOLO_PATH = "/content/weights (1).pt"

yolo_model = YOLO(YOLO_PATH)

print("YOLO model loaded.")

# ----------------------------------------------------------
# 2. UPLOAD INPUT DRONE IMAGE
# ----------------------------------------------------------

print("\nUpload the drone image to verify:")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No image uploaded.")

input_path = "/content/" + list(uploaded.keys())[0]

actual_image = cv2.imread(input_path)

if actual_image is None:
    raise ValueError("Unable to read uploaded image.")

print("Input image:", input_path)

# ----------------------------------------------------------
# 3. DETECT PROPELLERS
# ----------------------------------------------------------

def detect_propellers(image_path):

    results = yolo_model.predict(
        source=image_path,
        conf=0.25,
        verbose=False
    )

    result = results[0]
    detections = []

    for i, box in enumerate(result.boxes):

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        confidence = float(box.conf[0].cpu().numpy())

        center_x = float((x1 + x2) / 2)
        center_y = float((y1 + y2) / 2)

        detections.append({
            "id": f"D{i+1}",
            "center": [center_x, center_y],
            "bbox": [
                float(x1), float(y1),
                float(x2), float(y2)
            ],
            "confidence": confidence
        })

    return detections, result

detections, yolo_result = detect_propellers(input_path)

print("\nDETECTION RESULTS")
print("Propellers detected:", len(detections))

for d in detections:
    print(
        d["id"],
        "| Center:", d["center"],
        "| Confidence:", round(d["confidence"], 3)
    )

# Save annotated input image
annotated = yolo_result.plot()

cv2.imwrite(
    "/content/annotated_input.jpg",
    annotated
)

# ----------------------------------------------------------
# 4. LOAD PLUTO REFERENCE IMAGE
# ----------------------------------------------------------

reference_candidates = [
    "/content/pluto_reference_dataset/complete_drone/top/top_view_2.png",
    "/content/pluto_reference_dataset/complete_drone/top/reference.png"
]

reference_path = next(
    (p for p in reference_candidates if os.path.exists(p)),
    None
)

if reference_path is None:
    raise FileNotFoundError(
        "Pluto top-view reference image not found."
    )

reference_image = cv2.imread(reference_path)

if reference_image is None:
    raise ValueError("Cannot read Pluto reference image.")

# Detect propellers in reference image
reference_detections, reference_yolo_result = detect_propellers(
    reference_path
)

if len(reference_detections) != 4:
    raise ValueError(
        "Reference image must contain exactly four detected propellers. "
        f"Found: {len(reference_detections)}"
    )

print("\nREFERENCE IMAGE:", reference_path)
print("REFERENCE PROPELLERS:", len(reference_detections))

# ----------------------------------------------------------
# 5. GENERATE NORMALIZED REFERENCE MOTOR CENTERS
# ----------------------------------------------------------

reference_mapping = engine.map_image_coordinates(
    reference_detections,
    [0, -1]
)

if reference_mapping["status"] != "MAPPED":
    raise ValueError(
        "Could not map reference image: "
        + str(reference_mapping.get("errors"))
    )

reference_centers_normalized = {}

position_to_motor = engine.POSITION_TO_MOTOR

for detection_id, position in reference_mapping["mapping"].items():

    motor = position_to_motor[position]

    detection = next(
        d for d in reference_detections
        if d["id"] == detection_id
    )

    reference_centers_normalized[motor] = [
        detection["center"][0] / reference_image.shape[1],
        detection["center"][1] / reference_image.shape[0]
    ]

# ----------------------------------------------------------
# 6. RUN REFERENCE ENGINE ON INPUT
# ----------------------------------------------------------

engine_result = engine.run_reference_engine(
    REFERENCE_CONFIG,
    REFERENCE_GEOMETRY,
    detections,
    [0, -1]
)

print("\nREFERENCE ENGINE STATUS:", engine_result["status"])

print("MOTOR MAPPING:", engine_result.get("motor_mapping", {}))

# ----------------------------------------------------------
# 7. GENERATE DETAILED ERROR REPORT
# ----------------------------------------------------------

error_report = generate_error_report(
    detections=detections,
    engine_result=engine_result,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={},
    detected_types={}
)

# ----------------------------------------------------------
# 8. GENERATE CORRECTION REPORT
# ----------------------------------------------------------

correction_report = generate_correction_report(
    detections=detections,
    engine_result=engine_result,
    image_shape=actual_image.shape,
    reference_centers_normalized=reference_centers_normalized,
    reference_config=REFERENCE_CONFIG,
    appearance_scores={}
)

# ----------------------------------------------------------
# 9. DISPLAY BOTH REPORTS
# ----------------------------------------------------------

print_verification_reports(
    error_report,
    correction_report
)

# ----------------------------------------------------------
# 10. SAVE REPORTS
# ----------------------------------------------------------

with open("/content/detailed_error_report.json", "w") as f:
    json.dump(error_report, f, indent=4)

with open("/content/correction_report.json", "w") as f:
    json.dump(correction_report, f, indent=4)

with open("/content/drone_verification_summary.json", "w") as f:
    json.dump({
        "input_image": os.path.basename(input_path),
        "detected_propellers": len(detections),
        "engine_status": engine_result["status"],
        "motor_mapping": engine_result.get("motor_mapping", {}),
        "error_report": error_report,
        "correction_report": correction_report
    }, f, indent=4)

print("\n======================================")
print("ALL REPORTS GENERATED")
print("======================================")
print("/content/detailed_error_report.json")
print("/content/correction_report.json")
print("/content/drone_verification_summary.json")
print("/content/annotated_input.jpg")

# ----------------------------------------------------------
# 11. DOWNLOAD REPORTS
# ----------------------------------------------------------

files.download("/content/detailed_error_report.json")
files.download("/content/correction_report.json")
files.download("/content/drone_verification_summary.json")
